# Early levetiracetam prophylaxis and ICU delirium after acute brain injury — MIMIC-IV v3.1

**Stage 1: cohort build + data checks (aggregate output only).**

Data-use rule for this notebook: every cell prints **aggregate results only** (counts, percentages, summary stats).
Do not add cells that display patient-level rows if you plan to share output with an AI assistant — the PhysioNet DUA does not allow it.
Counts under 10 are masked as `<10`.

## 1. Authenticate and configure

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "YOUR_GCP_PROJECT"   # <-- your billing project (the one linked to PhysioNet)
WORK = "epi_lev_delirium"            # working dataset created in YOUR project

# MIMIC-IV on BigQuery. mimiciv_hosp / mimiciv_icu point to the current release (v3.1).
HOSP = "physionet-data.mimiciv_3_1_hosp"
ICU  = "physionet-data.mimiciv_3_1_icu"

client = bigquery.Client(project=PROJECT_ID)
client.create_dataset(f"{PROJECT_ID}.{WORK}", exists_ok=True)

def q(sql):
    return client.query(sql).to_dataframe()

def run(sql):
    client.query(sql).result()

def mask(df, min_n=10):
    import numpy as np
    out = df.copy().astype(object)
    for c in out.columns:
        out[c] = out[c].map(lambda x: "<10" if isinstance(x, (int, np.integer)) and 0 < x < min_n else x)
    return out
print("Ready.")

In [ ]:
# Repository paths (edit REPO if the repository is cloned elsewhere)
import os, io, pandas as pd
REPO = "/content/levetiracetam-delirium-triangulation"
GEN = "/content/lev_genomics"   # written by stage 5 (public summary statistics only)
def read_local(name): return pd.read_csv(f"{GEN}/{name}", sep="\t", compression="gzip", low_memory=False)
def run_script(name): exec(open(f"{REPO}/scripts/{name}").read(), globals())


## 2. Confirm dataset version
MIMIC-IV v3.x has 94,458 ICU stays.

In [ ]:
print([d.dataset_id for d in client.list_datasets("physionet-data") if "mimiciv" in d.dataset_id])
print(q(f"SELECT COUNT(*) AS n_icu_stays FROM `{ICU}.icustays`"))

## 3. Build the stage-1 cohort table

- **Acute brain injury (any diagnosis position):** TBI (ICD-9 800, 801, 803, 804, 850–854; ICD-10 S06), SAH (430; I60), ICH (431; I61)
- **Prior epilepsy:** epilepsy/status code (ICD-9 345; ICD-10 G40, G41) on any *earlier* admission
- **Exposure:** levetiracetam from eMAR (administered events) or ICU inputevents; `lev_48h` = first dose before ICU intime + 48 h
- Exclusions are kept as flags so the flowchart can be computed

In [ ]:
run(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{WORK}.cohort_stage1` AS
WITH abi_dx AS (
  SELECT hadm_id,
    MAX(IF((icd_version = 9 AND REGEXP_CONTAINS(icd_code, r'^(800|801|803|804|850|851|852|853|854)'))
        OR (icd_version = 10 AND STARTS_WITH(icd_code, 'S06')), 1, 0)) AS tbi,
    MAX(IF((icd_version = 9 AND STARTS_WITH(icd_code, '430'))
        OR (icd_version = 10 AND STARTS_WITH(icd_code, 'I60')), 1, 0)) AS sah,
    MAX(IF((icd_version = 9 AND STARTS_WITH(icd_code, '431'))
        OR (icd_version = 10 AND STARTS_WITH(icd_code, 'I61')), 1, 0)) AS ich,
    MAX(IF((icd_version = 9 AND STARTS_WITH(icd_code, '345'))
        OR (icd_version = 10 AND REGEXP_CONTAINS(icd_code, r'^G4[01]')), 1, 0)) AS epi_index_adm
  FROM `{HOSP}.diagnoses_icd`
  GROUP BY hadm_id
),
stays AS (
  SELECT i.subject_id, i.hadm_id, i.stay_id, i.intime, i.outtime, i.los, i.first_careunit,
         a.admittime,
         p.anchor_age + (EXTRACT(YEAR FROM a.admittime) - p.anchor_year) AS age,
         p.gender, p.anchor_year_group,
         d.tbi, d.sah, d.ich, d.epi_index_adm,
         ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) AS rn
  FROM `{ICU}.icustays` i
  JOIN `{HOSP}.admissions` a ON a.hadm_id = i.hadm_id
  JOIN `{HOSP}.patients`   p ON p.subject_id = i.subject_id
  JOIN abi_dx d ON d.hadm_id = i.hadm_id
  WHERE d.tbi + d.sah + d.ich > 0
),
prior_epi AS (
  SELECT DISTINCT s.stay_id
  FROM stays s
  JOIN `{HOSP}.admissions` a2
    ON a2.subject_id = s.subject_id AND a2.admittime < s.admittime
  JOIN `{HOSP}.diagnoses_icd` dx ON dx.hadm_id = a2.hadm_id
  WHERE (dx.icd_version = 9 AND STARTS_WITH(dx.icd_code, '345'))
     OR (dx.icd_version = 10 AND REGEXP_CONTAINS(dx.icd_code, r'^G4[01]'))
),
emar_any AS (
  SELECT DISTINCT hadm_id FROM `{HOSP}.emar`
),
lev_emar AS (
  SELECT e.hadm_id, MIN(e.charttime) AS first_lev
  FROM `{HOSP}.emar` e
  WHERE REGEXP_CONTAINS(LOWER(e.medication), r'levetiracetam|keppra')
    AND (LOWER(e.event_txt) LIKE '%administered%' OR e.event_txt IN ('Started', 'Restarted', 'Confirmed'))
  GROUP BY e.hadm_id
),
lev_ie AS (
  SELECT ie.stay_id, MIN(ie.starttime) AS first_lev
  FROM `{ICU}.inputevents` ie
  JOIN `{ICU}.d_items` di ON di.itemid = ie.itemid
  WHERE REGEXP_CONTAINS(LOWER(di.label), r'levetiracetam|keppra')
  GROUP BY ie.stay_id
),
other_asm AS (
  SELECT e.hadm_id, MIN(e.charttime) AS first_other_asm
  FROM `{HOSP}.emar` e
  WHERE REGEXP_CONTAINS(LOWER(e.medication), r'phenytoin|fosphenytoin|dilantin|lacosamide|vimpat|valpro|depakote|phenobarbital')
    AND (LOWER(e.event_txt) LIKE '%administered%' OR e.event_txt IN ('Started', 'Restarted', 'Confirmed'))
  GROUP BY e.hadm_id
)
SELECT s.*,
  (pe.stay_id IS NOT NULL) AS prior_epilepsy,
  (ea.hadm_id IS NOT NULL) AS has_emar,
  NULLIF(LEAST(COALESCE(le.first_lev, DATETIME '9999-12-31'),
               COALESCE(li.first_lev, DATETIME '9999-12-31')),
         DATETIME '9999-12-31') AS first_lev,
  oa.first_other_asm
FROM stays s
LEFT JOIN prior_epi pe ON pe.stay_id = s.stay_id
LEFT JOIN emar_any  ea ON ea.hadm_id = s.hadm_id
LEFT JOIN lev_emar  le ON le.hadm_id = s.hadm_id
LEFT JOIN lev_ie    li ON li.stay_id = s.stay_id
LEFT JOIN other_asm oa ON oa.hadm_id = s.hadm_id
""")

run(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{WORK}.cohort_stage1` AS
SELECT *,
  (first_lev IS NOT NULL AND first_lev <= DATETIME_ADD(intime, INTERVAL 48 HOUR)) AS lev_48h,
  (first_lev IS NOT NULL AND first_lev >  DATETIME_ADD(intime, INTERVAL 48 HOUR)) AS lev_late,
  (first_other_asm IS NOT NULL AND first_other_asm <= DATETIME_ADD(intime, INTERVAL 48 HOUR)) AS other_asm_48h,
  (rn = 1 AND age >= 18 AND NOT prior_epilepsy AND los >= 1) AS in_cohort
FROM `{PROJECT_ID}.{WORK}.cohort_stage1`
""")
print("cohort_stage1 built.")

## 4. Flowchart counts

In [ ]:
fc = q(f"""
SELECT
  COUNT(*)                                                        AS abi_icu_stays,
  COUNTIF(rn = 1)                                                 AS first_icu_stay,
  COUNTIF(rn = 1 AND age >= 18)                                   AS adult,
  COUNTIF(rn = 1 AND age >= 18 AND NOT prior_epilepsy)            AS no_prior_epilepsy,
  COUNTIF(in_cohort)                                              AS icu_los_ge_24h,
  COUNTIF(in_cohort AND has_emar)                                 AS with_emar_data
FROM `{PROJECT_ID}.{WORK}.cohort_stage1`
""")
print(mask(fc).T.to_string())

## 5. Exposure by injury type and by era
Checks eMAR coverage over time — eMAR is not populated for the earliest years, which would misclassify exposure.

In [ ]:
by_injury = q(f"""
SELECT
  CASE WHEN tbi = 1 AND sah + ich = 0 THEN 'TBI only'
       WHEN sah = 1 AND tbi + ich = 0 THEN 'SAH only'
       WHEN ich = 1 AND tbi + sah = 0 THEN 'ICH only'
       ELSE 'Multiple' END                         AS injury,
  COUNT(*)                                         AS n,
  COUNTIF(lev_48h)                                 AS lev_48h,
  COUNTIF(lev_late)                                AS lev_late,
  COUNTIF(NOT lev_48h AND NOT lev_late)            AS no_lev,
  COUNTIF(other_asm_48h)                           AS other_asm_48h,
  COUNTIF(epi_index_adm = 1)                       AS epilepsy_code_index_adm
FROM `{PROJECT_ID}.{WORK}.cohort_stage1`
WHERE in_cohort
GROUP BY injury ORDER BY n DESC
""")
print(mask(by_injury).to_string(index=False))

by_era = q(f"""
SELECT anchor_year_group,
  COUNT(*)            AS n,
  COUNTIF(has_emar)   AS with_emar,
  COUNTIF(lev_48h)    AS lev_48h
FROM `{PROJECT_ID}.{WORK}.cohort_stage1`
WHERE in_cohort
GROUP BY anchor_year_group ORDER BY anchor_year_group
""")
print()
print(mask(by_era).to_string(index=False))

## 6. Find delirium / sedation charting items in this cohort
Identifies CAM-ICU and RASS item IDs before defining the outcome (metadata + counts only).

In [ ]:
items = q(f"""
SELECT di.itemid, di.label, di.category,
       COUNT(*)                   AS n_rows,
       COUNT(DISTINCT ce.stay_id) AS n_stays
FROM `{ICU}.chartevents` ce
JOIN `{ICU}.d_items` di ON di.itemid = ce.itemid
JOIN `{PROJECT_ID}.{WORK}.cohort_stage1` c ON c.stay_id = ce.stay_id AND c.in_cohort
WHERE REGEXP_CONTAINS(LOWER(di.label), r'cam-icu|delirium|richmond|rass')
GROUP BY 1, 2, 3
ORDER BY n_stays DESC
""")
print(mask(items).to_string(index=False))

## 7. Value distribution of the delirium assessment items

In [ ]:
vals = q(f"""
SELECT di.label, ce.value, COUNT(*) AS n
FROM `{ICU}.chartevents` ce
JOIN `{ICU}.d_items` di ON di.itemid = ce.itemid
JOIN `{PROJECT_ID}.{WORK}.cohort_stage1` c ON c.stay_id = ce.stay_id AND c.in_cohort
WHERE REGEXP_CONTAINS(LOWER(di.label), r'cam-icu|delirium assessment')
GROUP BY 1, 2
ORDER BY 1, n DESC
""")
print(mask(vals).to_string(index=False))

**Next (stage 2):** once item IDs are confirmed — delirium outcome over days 1–7 with coma (RASS −4/−5) as a competing state, GCS/severity covariates, propensity score, and eICU replication.

In [ ]:
run_script("stage2_lev_delirium.py")

In [ ]:
run_script("stage3_ml_hte.py")

In [ ]:
run_script("stage4_network_pharm.py")

In [ ]:
run_script("stage5_get_genomic_data.py")   # downloads public GWAS / eQTL data and writes region extracts to GEN

In [ ]:
# Stage 5b — cis-eQTL exposure data from eQTL Catalogue FTP (tabix), then upload all genomic extracts to Drive.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysam"], check=False)
import os, io, requests, pysam, pandas as pd, numpy as np
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

OUT = "/content/lev_genomics"; os.makedirs(OUT, exist_ok=True)
GENES = {"SV2A": ("ENSG00000159164", "1", 149903318, 149918572),
         "CACNA1B": ("ENSG00000148408", "9", 137877782, 138124624)}
WIN = 1_000_000

meta = pd.read_csv("https://raw.githubusercontent.com/eQTL-Catalogue/eQTL-Catalogue-resources/HEAD/data_tables/dataset_metadata_r8.tsv", sep="\t")
print("metadata columns:", list(meta.columns))
ge = meta[meta.quant_method == "ge"]
brain = ge[ge.tissue_label.str.contains("brain|cortex|cerebell|hippocamp|caudate|putamen|accumbens|hypothalamus|amygdala|substantia|spinal", case=False, na=False)]
blood = ge[(ge.study_label == "GTEx") & ge.tissue_label.str.contains("blood", case=False, na=False)]
sel = pd.concat([brain, blood]).drop_duplicates("dataset_id")
print(f"Selected {len(sel)} datasets:", sel[["study_label", "tissue_label", "sample_size"]].to_string(index=False) if "sample_size" in sel else sel[["study_label", "tissue_label"]].to_string(index=False))

rows = []
for _, d in sel.iterrows():
    url = f"https://ftp.ebi.ac.uk/pub/databases/spot/eQTL/sumstats/{d.study_id}/{d.dataset_id}/{d.dataset_id}.all.tsv.gz"
    try:
        tb = pysam.TabixFile(url)
        header = None
        for g, (gid, chrom, s, e) in GENES.items():
            for cname in (chrom, f"chr{chrom}"):
                if cname in tb.contigs:
                    break
            recs = [r.split("\t") for r in tb.fetch(cname, max(0, s - WIN), e + WIN)]
            if not recs:
                continue
            if header is None:
                import zlib
                h = requests.get(url, headers={"Range": "bytes=0-65535"}, timeout=60).content
                header = zlib.decompressobj(16 + zlib.MAX_WBITS).decompress(h).decode(errors="ignore").split("\n")[0].split("\t")
            df_ = pd.DataFrame(recs, columns=header[:len(recs[0])])
            df_ = df_[df_["gene_id"] == gid]
            df_["gene"], df_["study"], df_["tissue"], df_["dataset_id"] = g, d.study_label, d.tissue_label, d.dataset_id
            rows.append(df_)
        print(f"  ok  {d.study_label:<12} {d.tissue_label[:40]}")
    except Exception as ex:
        print(f"  FAIL {d.study_label:<12} {d.tissue_label[:40]}: {str(ex)[:80]}")

eq = pd.concat(rows) if rows else pd.DataFrame()
if len(eq):
    for c in ["beta", "se", "pvalue", "maf", "an", "position"]:
        if c in eq:
            eq[c] = pd.to_numeric(eq[c], errors="coerce")
    eq["F"] = (eq["beta"] / eq["se"]) ** 2
    summ = (eq.groupby(["gene", "study", "tissue"])
              .agg(n_variants=("variant", "size"), min_p=("pvalue", "min"), max_F=("F", "max"),
                   n_p_lt5e8=("pvalue", lambda p: int((p < 5e-8).sum())),
                   n_p_lt1e4=("pvalue", lambda p: int((p < 1e-4).sum())))
              .reset_index().sort_values(["gene", "min_p"]))
    print("\nInstrument strength by gene and tissue:")
    print(summ.to_string(index=False))
    eq.to_csv(f"{OUT}/eqtl_catalogue_SV2A_CACNA1B.tsv.gz", sep="\t", index=False)
    summ.to_csv(f"{OUT}/eqtl_instrument_strength_summary.tsv", sep="\t", index=False)

# ---------------------------------------------------------------- upload everything to Drive
drv = build("drive", "v3")
q = "name='LEV_delirium_genomics' and mimeType='application/vnd.google-apps.folder' and trashed=false"
f = drv.files().list(q=q, fields="files(id)").execute().get("files", [])
fid = f[0]["id"] if f else drv.files().create(body={"name": "LEV_delirium_genomics",
                                                     "mimeType": "application/vnd.google-apps.folder"}, fields="id").execute()["id"]
print("\nUploading to Drive folder LEV_delirium_genomics:")
for fn in sorted(os.listdir(OUT)):
    p = f"{OUT}/{fn}"
    if fn.endswith(".tbl.gz") or os.path.getsize(p) > 300e6:
        print(f"  skip {fn} (raw/large)"); continue
    drv.files().create(body={"name": fn, "parents": [fid]}, media_body=MediaFileUpload(p, resumable=True), fields="id").execute()
    print(f"  uploaded {fn} ({os.path.getsize(p)/1e6:.1f} MB)")
print("Drive folder id:", fid)


In [ ]:
import requests, pandas as pd
tree = requests.get('https://api.github.com/repos/eQTL-Catalogue/eQTL-Catalogue-resources/git/trees/HEAD?recursive=1', timeout=60).json()['tree']
tsv=[t['path'] for t in tree if t['path'].endswith('.tsv')]
print('PROBE_START')
print(tsv)
for p in tsv:
    if 'dataset' in p.lower() or 'tabix' in p.lower():
        d=pd.read_csv('https://raw.githubusercontent.com/eQTL-Catalogue/eQTL-Catalogue-resources/HEAD/'+p, sep='\t', nrows=3)
        print(p, list(d.columns)); print(d.head(2).to_string())
print('PROBE_END')

In [ ]:
# Stage 6 — Drug-target Mendelian randomization + colocalization: SV2A / CACNA1B expression -> delirium
# Public summary statistics only. Runs in Colab after stage 5/5b (files in /content/lev_genomics).
#
# Exposure : cis-eQTLs (eQTL Catalogue r8; brain tissues), effect allele = alt, beta on normalised expression scale
# Outcome  : (discovery) Raptis & Tenesa 2025 delirium meta-analysis (UKB + AoU); (replication) FinnGen R12 F5_DELIRIUM
# Methods  : (1) Wald ratio using the lead cis-eQTL per dataset (p < 5e-8; F > 10), delta-method SE
#            (2) Colocalization (coloc.abf; Giambartolomei 2014) over gene +/- 500 kb, priors p1=p2=1e-4, p12=1e-5
#            (3) Positive control: same pipeline across every gene with a lead variant genome-wide significant for delirium
#                is not feasible without LD; instead report the APOE region lead as an outcome-side sanity check.

import os, numpy as np, pandas as pd
from scipy import stats

D = "/content/lev_genomics"
GENES = {"SV2A": ("1", 149903318, 149918572), "CACNA1B": ("9", 137877782, 138124624)}
COLOC_WIN = 500_000
P1, P2, P12 = 1e-4, 1e-4, 1e-5


def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


# ---------------------------------------------------------------- load
eq = pd.read_csv(f"{D}/eqtl_catalogue_SV2A_CACNA1B.tsv.gz", sep="\t", low_memory=False)
meta = pd.read_csv(f"{D}/delirium_meta_SV2A_CACNA1B_region.tsv.gz", sep="\t", low_memory=False)
fgf = [f for f in os.listdir(D) if f.startswith("finngen_") and f.endswith("_region.tsv.gz")]
fg = pd.read_csv(f"{D}/{fgf[0]}", sep="\t", low_memory=False) if fgf else pd.DataFrame()

for c in ["position", "beta", "se", "pvalue", "maf"]:
    eq[c] = pd.to_numeric(eq[c], errors="coerce")
eq["chrom"] = eq["chromosome"].astype(str).str.replace("chr", "")
eq = eq.dropna(subset=["beta", "se", "position"])
eq["key"] = eq["chrom"] + ":" + eq["position"].astype(int).astype(str)

# outcome tables in a common format: key, ea (effect allele), oa, beta, se, p, eaf
m = pd.DataFrame({
    "key": meta["Chr"].astype(str).str.replace("chr", "") + ":" + meta["Pos"].astype(int).astype(str),
    "ea": meta["Allele1"].str.upper(), "oa": meta["Allele2"].str.upper(),
    "beta": pd.to_numeric(meta["Effect"], errors="coerce"), "se": pd.to_numeric(meta["StdErr"], errors="coerce"),
    "p": pd.to_numeric(meta["P_sterr"], errors="coerce"), "eaf": pd.to_numeric(meta["Freq1"], errors="coerce"),
    "N": pd.to_numeric(meta["N"], errors="coerce")}).dropna(subset=["beta", "se"])
OUT = {"Delirium meta (UKB+AoU)": m}
if len(fg):
    f2 = pd.DataFrame({
        "key": fg["#chrom"].astype(str).str.replace("chr", "") + ":" + fg["pos"].astype(int).astype(str),
        "ea": fg["alt"].str.upper(), "oa": fg["ref"].str.upper(),
        "beta": pd.to_numeric(fg["beta"], errors="coerce"), "se": pd.to_numeric(fg["sebeta"], errors="coerce"),
        "p": pd.to_numeric(fg["pval"], errors="coerce"), "eaf": pd.to_numeric(fg["af_alt"], errors="coerce")}).dropna(subset=["beta", "se"])
    OUT["FinnGen R12"] = f2


def harmonise(e, o):
    """Align outcome effect to the eQTL alt allele. Drops allele mismatches and ambiguous palindromes (MAF > 0.42)."""
    x = e.merge(o, on="key", suffixes=("_e", "_o"))
    same = (x.alt == x.ea) & (x.ref == x.oa)
    flip = (x.alt == x.oa) & (x.ref == x.ea)
    x = x[same | flip].copy()
    x.loc[flip[same | flip].values, "beta_o"] = -x.loc[flip[same | flip].values, "beta_o"]
    pal = x.apply(lambda r: {r.ref, r.alt} in ({"A", "T"}, {"C", "G"}), axis=1)
    x = x[~(pal & (x.maf > 0.42))]
    return x


def log_abf(beta, se, W):
    V = se ** 2; z = beta / se; r = W ** 2 / (W ** 2 + V)
    return 0.5 * (np.log(1 - r) + r * z ** 2)


def logsum(v):
    mx = np.max(v); return mx + np.log(np.sum(np.exp(v - mx)))


def coloc_abf(l1, l2):
    lH1, lH2 = np.log(P1) + logsum(l1), np.log(P2) + logsum(l2)
    lH3 = np.log(P1) + np.log(P2) + (logsum(l1) + logsum(l2) + np.log1p(-np.exp(min(0, logsum(l1 + l2) - logsum(l1) - logsum(l2)))))
    lH4 = np.log(P12) + logsum(l1 + l2)
    v = np.array([0.0, lH1, lH2, lH3, lH4]); return np.exp(v - logsum(v))


# ---------------------------------------------------------------- 1. instrument strength
section("1. Cis-eQTL instruments (lead variant per dataset)")
inst_rows = []
for (g, study, tissue, dsid), s in eq.groupby(["gene", "study", "tissue", "dataset_id"]):
    s = s[s.molecular_trait_id == s.molecular_trait_id.iloc[0]] if "molecular_trait_id" in s else s
    lead = s.loc[s.pvalue.idxmin()]
    inst_rows.append(dict(gene=g, study=study, tissue=tissue, dataset_id=dsid, n_cis=len(s),
                          lead_variant=lead.variant, rsid=lead.get("rsid", ""), lead_p=lead.pvalue,
                          beta_exp=lead.beta, se_exp=lead.se, F=(lead.beta / lead.se) ** 2,
                          usable=(lead.pvalue < 5e-8) and ((lead.beta / lead.se) ** 2 > 10)))
inst = pd.DataFrame(inst_rows).sort_values(["gene", "lead_p"])
print(inst[["gene", "study", "tissue", "n_cis", "rsid", "lead_p", "F", "usable"]].to_string(index=False))

# ---------------------------------------------------------------- 2. Wald ratio MR
section("2. Wald-ratio MR: OR for delirium per 1-SD genetically higher expression")
mr_rows = []
for _, r in inst[inst.usable].iterrows():
    e_lead = eq[(eq.dataset_id == r.dataset_id) & (eq.gene == r.gene) & (eq.variant == r.lead_variant)].head(1)
    for oname, o in OUT.items():
        x = harmonise(e_lead[["key", "ref", "alt", "beta", "se", "maf"]], o)
        if not len(x):
            mr_rows.append(dict(gene=r.gene, study=r.study, tissue=r.tissue, outcome=oname, note="lead variant absent/ambiguous in outcome"))
            continue
        x = x.iloc[0]
        wr = x.beta_o / x.beta_e
        se = abs(x.se_o / x.beta_e)
        p = 2 * stats.norm.sf(abs(wr / se))
        mr_rows.append(dict(gene=r.gene, study=r.study, tissue=r.tissue, outcome=oname, rsid=r.rsid,
                            OR=np.exp(wr), lo=np.exp(wr - 1.96 * se), hi=np.exp(wr + 1.96 * se), p=p,
                            outcome_p_at_lead=x.p))
mr = pd.DataFrame(mr_rows)
if len(mr) and "OR" in mr:
    mr["OR (95% CI)"] = mr.apply(lambda r: f"{r.OR:.3f} ({r.lo:.3f}-{r.hi:.3f})" if pd.notna(r.get("OR")) else r.get("note"), axis=1)
    k = mr.dropna(subset=["p"])
    mr["p_bonf"] = np.minimum(mr["p"] * max(len(k), 1), 1.0)
    print(mr[["gene", "study", "tissue", "outcome", "rsid", "OR (95% CI)", "p", "p_bonf"]].to_string(index=False))
    print(f"\nTests: {len(k)}; Bonferroni threshold p < {0.05/max(len(k),1):.2e}")

    # fixed-effect summary per gene x outcome across tissues (NOT independent; descriptive only)
    print("\nDescriptive pooled estimate across tissues (correlated instruments; interpret with caution):")
    for (g, oname), s in k.groupby(["gene", "outcome"]):
        b = np.log(s.OR); se = (np.log(s.hi) - np.log(s.lo)) / 3.92; w = 1 / se ** 2
        bb = np.sum(w * b) / np.sum(w); ss = np.sqrt(1 / np.sum(w))
        print(f"  {g:<8} {oname:<24} OR {np.exp(bb):.3f} ({np.exp(bb-1.96*ss):.3f}-{np.exp(bb+1.96*ss):.3f})  across {len(s)} tissues")
else:
    print("No usable instruments (no dataset with lead cis-eQTL p < 5e-8 and F > 10).")

# ---------------------------------------------------------------- 3. Colocalization
section(f"3. Colocalization (coloc.abf, gene +/- {COLOC_WIN//1000} kb; priors p1=p2=1e-4, p12=1e-5)")
co_rows = []
for (g, study, tissue, dsid), s in eq.groupby(["gene", "study", "tissue", "dataset_id"]):
    chrom, gs, ge_ = GENES[g]
    s = s[(s.position >= gs - COLOC_WIN) & (s.position <= ge_ + COLOC_WIN)]
    for oname, o in OUT.items():
        x = harmonise(s[["key", "ref", "alt", "beta", "se", "maf"]], o).drop_duplicates("key")
        if len(x) < 50:
            continue
        pp = coloc_abf(log_abf(x.beta_e.values, x.se_e.values, 0.15), log_abf(x.beta_o.values, x.se_o.values, 0.2))
        co_rows.append(dict(gene=g, study=study, tissue=tissue, outcome=oname, n_snps=len(x),
                            PP_H0=pp[0], PP_H1=pp[1], PP_H2=pp[2], PP_H3=pp[3], PP_H4=pp[4]))
co = pd.DataFrame(co_rows)
if len(co):
    co = co.sort_values(["gene", "outcome", "PP_H4"], ascending=[True, True, False])
    print(co.round(3).to_string(index=False))
    print("\nInterpretation: PP.H4 > 0.8 = shared causal variant; PP.H1 high = eQTL signal only (no delirium signal);")
    print("PP.H3 high = distinct causal variants.")

# ---------------------------------------------------------------- 4. outcome-side sanity check
section("4. Outcome data sanity check")
for oname, o in OUT.items():
    print(f"{oname}: {len(o):,} variants in SV2A/CACNA1B windows; median SE {o.se.median():.4f}; "
          f"min p {o.p.min():.2e}" + (f"; median N {o.N.median():,.0f}" if "N" in o else ""))
gw = f"{D}/delirium_meta_gwsig_hits.tsv.gz"
if os.path.exists(gw):
    try:
        h = pd.read_csv(gw, sep="\t")
        top = h.sort_values("P_sterr").head(5)[["Chr", "Pos", "MarkerName", "rsid", "Effect", "P_sterr"]]
        print("\nTop genome-wide delirium hits (positive control: APOE locus chr19 ~44.9 Mb expected):")
        print(top.to_string(index=False))
    except Exception as ex:
        print("gw hits not parsed:", ex)

# ---------------------------------------------------------------- save
inst.to_csv(f"{D}/results_mr_instruments.tsv", sep="\t", index=False)
if len(mr): mr.to_csv(f"{D}/results_mr_wald.tsv", sep="\t", index=False)
if len(co): co.to_csv(f"{D}/results_coloc.tsv", sep="\t", index=False)
print("\nSaved results_* tables to", D)


In [ ]:
# Stage 4b (corrected delirium module) — Bioinformatics / network-pharmacology layer (public data only; no MIMIC data used here)
# 1) Drug targets (Open Targets / ChEMBL mechanisms of action) for levetiracetam and comparator ASMs
# 2) Delirium-associated genes (Open Targets association scores)
# 3) Network proximity of each drug's targets to the delirium module in the STRING v12 human interactome
#    (closest-distance d_c with degree-preserving randomisation; Guney et al., Nat Commun 2016)
# 4) Shortest-path "bridge" genes between SV2A and delirium genes + pathway enrichment (Enrichr)
# 5) Tissue specificity of SV2A (GTEx v8 median TPM)

import subprocess, sys, io, gzip, json, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gseapy"], check=False)
import requests, numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

OT = "https://api.platform.opentargets.org/api/v4/graphql"
rng = np.random.default_rng(2026)
N_RAND = 1000


def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def gql(query, variables=None):
    r = requests.post(OT, json={"query": query, "variables": variables or {}}, timeout=60)
    r.raise_for_status()
    return r.json()["data"]


def search(term, entity):
    d = gql("""query($q:String!,$e:[String!]!){search(queryString:$q,entityNames:$e,page:{index:0,size:5}){hits{id name entity}}}""",
            {"q": term, "e": [entity]})
    return d["search"]["hits"]


# ---------------------------------------------------------------- 1. drug targets
section("1. Drug targets (Open Targets / ChEMBL mechanism of action)")
DRUGS = ["levetiracetam", "brivaracetam", "phenytoin", "lacosamide", "valproic acid", "lamotrigine"]
drug_targets = {}
for name in DRUGS:
    try:
        hit = search(name, "drug")[0]
        d = gql("""query($id:String!){drug(chemblId:$id){name mechanismsOfAction{rows{mechanismOfAction targets{approvedSymbol}}}}}""",
                {"id": hit["id"]})
        rows = d["drug"]["mechanismsOfAction"]["rows"]
        syms = sorted({t["approvedSymbol"] for r in rows for t in r["targets"]})
        moa = sorted({r["mechanismOfAction"] for r in rows})
        drug_targets[name] = syms
        print(f"{name:<14} {hit['id']:<14} targets={len(syms):>2}  {', '.join(syms[:12])}{' ...' if len(syms) > 12 else ''}")
        print(f"{'':<14} MoA: {'; '.join(moa)[:150]}")
    except Exception as ex:
        print(f"{name}: lookup failed ({ex})")

# ---------------------------------------------------------------- 2. delirium genes (genetic + literature evidence ONLY)
section("2. Delirium-associated genes (Open Targets; genetic-association and literature evidence only)")
dis_hits = search("delirium", "disease")
print("Disease hits:", [(h["id"], h["name"]) for h in dis_hits])
rows_ = []
for h in dis_hits:
    if "delirium" not in h["name"].lower() or "alcohol" in h["name"].lower():
        continue
    d = gql("""query($id:String!){disease(efoId:$id){name associatedTargets(page:{index:0,size:3000}){count rows{score datatypeScores{id score} target{approvedSymbol}}}}}""",
            {"id": h["id"]})
    for r in d["disease"]["associatedTargets"]["rows"]:
        ds_ = {x["id"]: x["score"] for x in r["datatypeScores"]}
        rows_.append(dict(gene=r["target"]["approvedSymbol"], disease=h["name"], overall=r["score"], **ds_))
ev = pd.DataFrame(rows_).fillna(0)
for c in ["genetic_association", "literature", "known_drug", "animal_model", "rna_expression", "affected_pathway"]:
    if c not in ev:
        ev[c] = 0.0
ev = ev.groupby("gene")[["genetic_association", "literature", "known_drug", "animal_model", "rna_expression", "affected_pathway", "overall"]].max()
print(f"All associated targets: {len(ev)}; with known_drug evidence: {(ev.known_drug>0).sum()}")
ev["gen_lit"] = ev[["genetic_association", "literature"]].max(axis=1)
DELIRIUM = list(ev[ev.gen_lit >= 0.05].sort_values("gen_lit", ascending=False).index)
excluded_drug_only = ev[(ev.known_drug > 0) & (ev.gen_lit < 0.05)].index
print(f"Delirium module (genetic/literature score >= 0.05): {len(DELIRIUM)} genes; "
      f"excluded {len(excluded_drug_only)} drug-evidence-only genes")
print("Module:", ", ".join(DELIRIUM))
for g in ["SV2A", "CACNA1B"]:
    print(f"  {g} in module: {g in DELIRIUM}")
SENS = list(ev[ev.genetic_association >= 0.05].index)
print(f"Sensitivity module (genetic association only, >= 0.05): {len(SENS)} genes: {', '.join(SENS)}")

# ---------------------------------------------------------------- 3. STRING interactome
section("3. STRING v12 human interactome (combined score >= 700)")
base = "https://stringdb-downloads.org/download"
info = pd.read_csv(io.BytesIO(requests.get(f"{base}/protein.info.v12.0/9606.protein.info.v12.0.txt.gz", timeout=300).content),
                   sep="\t", compression="gzip", usecols=[0, 1])
info.columns = ["pid", "symbol"]
links = pd.read_csv(io.BytesIO(requests.get(f"{base}/protein.links.v12.0/9606.protein.links.v12.0.txt.gz", timeout=600).content),
                    sep=" ", compression="gzip")
links = links[links.combined_score >= 700]
p2s = dict(zip(info.pid, info.symbol))
links["a"], links["b"] = links.protein1.map(p2s), links.protein2.map(p2s)
nodes = pd.Index(sorted(set(links.a) | set(links.b)))
ai, bi = nodes.get_indexer(links.a), nodes.get_indexer(links.b)
A = csr_matrix((np.ones(len(ai)), (ai, bi)), shape=(len(nodes), len(nodes)))
A = ((A + A.T) > 0).astype(np.int8)
ncomp, lab = connected_components(A, directed=False)
lcc_label = np.bincount(lab).argmax()
keep = np.where(lab == lcc_label)[0]
A = A[keep][:, keep]; nodes = nodes[keep]
deg = np.asarray(A.sum(1)).ravel()
idx = {s: i for i, s in enumerate(nodes)}
print(f"Largest connected component: {len(nodes):,} proteins, {A.nnz // 2:,} interactions")

D_idx = np.array([idx[g] for g in DELIRIUM if g in idx])
D_sens = np.array([idx[g] for g in SENS if g in idx])
print(f"Delirium genes mapped to interactome: {len(D_idx)}/{len(DELIRIUM)}; sensitivity module: {len(D_sens)}/{len(SENS)}")

# degree bins (>=100 nodes per bin) for degree-preserving randomisation
order = np.argsort(deg); bins, cur = [], []
for i in order:
    cur.append(i)
    if len(cur) >= 100 and (len(bins) == 0 or True):
        bins.append(np.array(cur)); cur = []
if cur:
    bins[-1] = np.concatenate([bins[-1], cur])
bin_of = np.empty(len(nodes), dtype=int)
for b, members in enumerate(bins):
    bin_of[members] = b


def rand_like(S):
    return np.array([rng.choice(bins[bin_of[s]]) for s in S])


def d_closest(T, D):
    dist = shortest_path(A, directed=False, unweighted=True, indices=T)
    return np.mean(dist[:, D].min(axis=1))


section(f"4. Network proximity to delirium module (closest distance, {N_RAND} degree-matched randomisations)")
rows = []
for modname, DM in [("gen+lit", D_idx), ("genetic only", D_sens)]:
  if len(DM) < 5:
    print(f"{modname}: module too small ({len(DM)}) - skipped"); continue
  for name, tg in drug_targets.items():
    T_idx = np.array([idx[g] for g in tg if g in idx])
    if len(T_idx) == 0:
        rows.append([modname, name, 0, np.nan, np.nan, np.nan, np.nan]); continue
    obs = d_closest(T_idx, DM)
    null = np.array([d_closest(rand_like(T_idx), rand_like(DM)) for _ in range(N_RAND)])
    z = (obs - null.mean()) / null.std(ddof=1)
    p = (np.sum(null <= obs) + 1) / (N_RAND + 1)
    rows.append([modname, name, len(T_idx), round(obs, 3), round(null.mean(), 3), round(z, 2), round(p, 4)])
prox = pd.DataFrame(rows, columns=["module", "drug", "targets_in_network", "d_c", "null_mean", "z", "p_empirical"])
print(prox.to_string(index=False))
print("Interpretation: z < -1.5 and p < 0.05 indicate targets closer to the delirium module than expected by chance.")

# ---------------------------------------------------------------- 5. bridge genes SV2A -> delirium
section("5. Shortest-path bridge genes from levetiracetam targets to delirium genes")
lev_T = [idx[g] for g in drug_targets.get("levetiracetam", []) if g in idx]
bridge = set()
if lev_T:
    dist, pred = shortest_path(A, directed=False, unweighted=True, indices=lev_T, return_predecessors=True)
    for k, t in enumerate(lev_T):
        dmin = dist[k, D_idx].min()
        for dgi in D_idx[dist[k, D_idx] == dmin]:
            cur = dgi
            while cur != t and cur >= 0:
                bridge.add(nodes[cur]); cur = pred[k, cur]
            bridge.add(nodes[t])
        print(f"{nodes[t]}: nearest delirium gene(s) at distance {int(dmin)} -> "
              f"{', '.join(nodes[D_idx[dist[k, D_idx] == dmin]][:15])}")
print(f"Bridge module ({len(bridge)} genes): {', '.join(sorted(bridge))}")

section("6. Pathway enrichment of bridge module (Enrichr)")
try:
    import gseapy as gp
    if len(bridge) >= 5:
        enr = gp.enrichr(gene_list=sorted(bridge),
                         gene_sets=["GO_Biological_Process_2023", "KEGG_2021_Human", "Reactome_2022"],
                         organism="human", outdir=None, cutoff=1.0)
        res = enr.results.sort_values("Adjusted P-value").head(15)
        res["Term"] = res["Term"].str.slice(0, 70)
        print(res[["Gene_set", "Term", "Overlap", "Adjusted P-value"]].to_string(index=False))
    else:
        print("Bridge module too small for enrichment.")
except Exception as ex:
    print(f"Enrichment failed: {ex}")

prox.to_csv("/content/lev_genomics/results_network_proximity.tsv", sep="\t", index=False)
ev.to_csv("/content/lev_genomics/results_delirium_gene_evidence.tsv", sep="\t")
print("\nDone (public data only).")


In [ ]:
# Restore query helpers (overwritten by later stages)
import numpy as np, pandas as pd
def q(sql):
    return client.query(sql).to_dataframe()
def run(sql):
    client.query(sql).result()
def mask(df, min_n=10):
    out = df.copy().astype(object)
    for c in out.columns:
        out[c] = out[c].map(lambda x: '<10' if isinstance(x, (int, np.integer)) and 0 < x < min_n else x)
    return out
print('helpers restored')

In [ ]:
# Stage 7a — eICU-CRD v2.0 probe (aggregate counts only; no patient-level rows printed)
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 80)
PROJECT_ID = "YOUR_GCP_PROJECT"
E = "physionet-data.eicu_crd"
client = bigquery.Client(project=PROJECT_ID)
def q(sql): return client.query(sql).to_dataframe()
def mask(df, min_n=10):
    import numpy as np
    out = df.copy().astype(object)
    for c in out.columns:
        out[c] = out[c].map(lambda x: "<10" if isinstance(x, (int, np.integer)) and 0 < x < min_n else x)
    return out

print("A. access:", q(f"SELECT COUNT(*) n_stays, COUNT(DISTINCT hospitalid) n_hosp FROM `{E}.patient`").to_dict("records"))

print("\nB. brain-injury APACHE admission diagnoses")
print(mask(q(f"""SELECT apacheadmissiondx, COUNT(*) n FROM `{E}.patient`
WHERE REGEXP_CONTAINS(LOWER(apacheadmissiondx), r'intracranial|subarachnoid|head only|head/multiple|subdural|epidural|hemorrhage/hematoma')
GROUP BY 1 ORDER BY n DESC""")).to_string(index=False))

print("\nC. delirium charting labels/values (nurseCharting)")
print(mask(q(f"""SELECT nursingchartcelltypevallabel lab, nursingchartcelltypevalname nm, nursingchartvalue val, COUNT(*) n
FROM `{E}.nursecharting`
WHERE REGEXP_CONTAINS(LOWER(nursingchartcelltypevallabel), r'delirium')
GROUP BY 1,2,3 ORDER BY n DESC LIMIT 40""")).to_string(index=False))

print("\nD. sedation (RASS) labels")
print(mask(q(f"""SELECT nursingchartcelltypevallabel lab, nursingchartcelltypevalname nm, COUNT(*) n
FROM `{E}.nursecharting`
WHERE REGEXP_CONTAINS(LOWER(nursingchartcelltypevallabel), r'sedation|rass')
GROUP BY 1,2 ORDER BY n DESC LIMIT 15""")).to_string(index=False))

print("\nE. levetiracetam / other ASM in medication table")
print(mask(q(f"""SELECT REGEXP_EXTRACT(LOWER(drugname), r'(levetiracetam|keppra|phenytoin|fosphenytoin|dilantin|cerebyx|valpro|depakote|depacon|lacosamide|vimpat|phenobarbital|lamotrigine|carbamazepine|oxcarbazepine|topiramate|brivaracetam|zonisamide)') drug,
  drugordercancelled, COUNT(*) n, COUNT(DISTINCT patientunitstayid) stays, COUNT(DISTINCT p.hospitalid) hosp
FROM `{E}.medication` m JOIN `{E}.patient` p USING (patientunitstayid)
WHERE REGEXP_CONTAINS(LOWER(drugname), r'levetiracetam|keppra|phenytoin|fosphenytoin|dilantin|cerebyx|valpro|depakote|depacon|lacosamide|vimpat|phenobarbital|lamotrigine|carbamazepine|oxcarbazepine|topiramate|brivaracetam|zonisamide')
GROUP BY 1,2 ORDER BY n DESC""")).to_string(index=False))

print("\nF. hospitals with both CAM-ICU-type delirium charting and medication data, brain-injury stays")
print(mask(q(f"""WITH abi AS (SELECT patientunitstayid, hospitalid FROM `{E}.patient`
  WHERE REGEXP_CONTAINS(LOWER(apacheadmissiondx), r'intracranial|subarachnoid|head only|head/multiple|subdural|epidural')),
 del AS (SELECT DISTINCT patientunitstayid FROM `{E}.nursecharting` WHERE REGEXP_CONTAINS(LOWER(nursingchartcelltypevallabel), r'delirium')),
 med AS (SELECT DISTINCT patientunitstayid FROM `{E}.medication`)
SELECT COUNT(*) abi_stays, COUNTIF(d.patientunitstayid IS NOT NULL) with_delirium_charting,
  COUNTIF(m.patientunitstayid IS NOT NULL) with_meds,
  COUNTIF(d.patientunitstayid IS NOT NULL AND m.patientunitstayid IS NOT NULL) both,
  COUNT(DISTINCT IF(d.patientunitstayid IS NOT NULL AND m.patientunitstayid IS NOT NULL, a.hospitalid, NULL)) hosp_both
FROM abi a LEFT JOIN del d USING (patientunitstayid) LEFT JOIN med m USING (patientunitstayid)""")).to_string(index=False))

print("\nG. seizure/epilepsy history and diagnosis strings (counts)")
print(mask(q(f"""SELECT 'pasthistory' src, COUNT(DISTINCT patientunitstayid) stays FROM `{E}.pasthistory`
  WHERE REGEXP_CONTAINS(LOWER(pasthistorypath), r'seizure|epilep')
UNION ALL SELECT 'diagnosis', COUNT(DISTINCT patientunitstayid) FROM `{E}.diagnosis`
  WHERE REGEXP_CONTAINS(LOWER(diagnosisstring), r'seizure|epilep|status epilepticus')""")).to_string(index=False))


In [ ]:
# Stage 7b — eICU-CRD v2.0 external replication: early levetiracetam and delirium after acute brain injury
# Mirrors the MIMIC-IV design. Patient-level data stay in this runtime; ONLY aggregates are printed (counts <10 masked).
import numpy as np, pandas as pd, statsmodels.api as sm, warnings
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

ABI = r"intracranial|subarachnoid|head only|head/multiple|subdural|epidural"
ASM_OTHER = r"phenytoin|fosphenytoin|dilantin|cerebyx|valpro|depakote|depacon|lacosamide|vimpat|phenobarbital|lamotrigine|carbamazepine|tegretol|oxcarbazepine|trileptal|topiramate|topamax|brivaracetam|briviact|zonisamide|pentobarbital"
LEV = r"levetiracetam|keppra"

sql = f"""
WITH base AS (
  SELECT p.patientunitstayid id, p.hospitalid, p.uniquepid, p.gender, p.age, p.apacheadmissiondx dx, p.unittype,
         p.unitdischargeoffset los_min, p.hospitaldischargestatus hstat, p.hospitaldischargeyear yr, p.unitvisitnumber,
         h.region, h.teachingstatus
  FROM `{E}.patient` p LEFT JOIN `{E}.hospital` h USING (hospitalid)
  WHERE REGEXP_CONTAINS(LOWER(p.apacheadmissiondx), r'{ABI}')),
del_hosp AS (SELECT DISTINCT p.hospitalid FROM `{E}.nursecharting` n JOIN `{E}.patient` p USING (patientunitstayid)
  WHERE n.nursingchartcelltypevalname = 'Delirium Score'),
anymed AS (SELECT DISTINCT patientunitstayid id FROM `{E}.medication`),
lev AS (SELECT patientunitstayid id, MIN(drugstartoffset) lev_start FROM `{E}.medication`
  WHERE REGEXP_CONTAINS(LOWER(drugname), r'{LEV}') AND drugordercancelled = 'No' AND drugstartoffset >= -1440 GROUP BY 1),
asm AS (SELECT patientunitstayid id, MIN(start) asm_start FROM (
    SELECT patientunitstayid, drugstartoffset start FROM `{E}.medication`
      WHERE REGEXP_CONTAINS(LOWER(drugname), r'{ASM_OTHER}') AND drugordercancelled = 'No'
    UNION ALL SELECT patientunitstayid, infusionoffset FROM `{E}.infusiondrug`
      WHERE REGEXP_CONTAINS(LOWER(drugname), r'{ASM_OTHER}')) GROUP BY 1),
sz_hx AS (SELECT DISTINCT patientunitstayid id FROM `{E}.pasthistory` WHERE REGEXP_CONTAINS(LOWER(pasthistorypath), r'seizure|epilep')),
sz_dx AS (SELECT DISTINCT patientunitstayid id FROM `{E}.diagnosis`
  WHERE (REGEXP_CONTAINS(LOWER(diagnosisstring), r'seizure|epilep|status epilepticus') OR REGEXP_CONTAINS(icd9code, r'^(345|780\\.3)'))
    AND diagnosisoffset <= 2880),
dl AS (SELECT patientunitstayid id, nursingchartoffset t,
    CASE WHEN UPPER(TRIM(nursingchartvalue)) = 'YES' OR SAFE_CAST(nursingchartvalue AS INT64) >= 4 THEN 1
         WHEN UPPER(TRIM(nursingchartvalue)) = 'NO' OR SAFE_CAST(nursingchartvalue AS INT64) BETWEEN 0 AND 3 THEN 0 END pos
  FROM `{E}.nursecharting` WHERE nursingchartcelltypevalname = 'Delirium Score' AND patientunitstayid IN (SELECT id FROM base)),
sym AS (SELECT patientunitstayid id, nursingchartoffset t,
    CASE WHEN UPPER(TRIM(nursingchartvalue)) = 'YES' THEN 1 WHEN UPPER(TRIM(nursingchartvalue)) = 'NO' THEN 0 END pos
  FROM `{E}.nursecharting` WHERE nursingchartcelltypevallabel = 'Symptoms of Delirium Present' AND patientunitstayid IN (SELECT id FROM base)),
rass_raw AS (
  SELECT patientunitstayid id, nursingchartoffset t, SAFE_CAST(nursingchartvalue AS FLOAT64) v FROM `{E}.nursecharting`
    WHERE nursingchartcelltypevallabel = 'RASS' AND patientunitstayid IN (SELECT id FROM base)
  UNION ALL
  SELECT s.patientunitstayid, s.nursingchartoffset, SAFE_CAST(s.nursingchartvalue AS FLOAT64) FROM `{E}.nursecharting` s
    JOIN `{E}.nursecharting` k ON k.patientunitstayid = s.patientunitstayid AND k.nursingchartoffset = s.nursingchartoffset
     AND k.nursingchartcelltypevalname = 'Sedation Scale' AND UPPER(k.nursingchartvalue) LIKE 'RASS%'
    WHERE s.nursingchartcelltypevalname = 'Sedation Score' AND s.patientunitstayid IN (SELECT id FROM base)
      AND k.patientunitstayid IN (SELECT id FROM base)),
inf AS (SELECT patientunitstayid id,
    MAX(IF(REGEXP_CONTAINS(LOWER(drugname), r'propofol|diprivan'), 1, 0)) propofol,
    MAX(IF(REGEXP_CONTAINS(LOWER(drugname), r'midazolam|versed|lorazepam|ativan'), 1, 0)) benzo,
    MAX(IF(REGEXP_CONTAINS(LOWER(drugname), r'dexmedetomidine|precedex'), 1, 0)) dex,
    MAX(IF(REGEXP_CONTAINS(LOWER(drugname), r'fentanyl|morphine|hydromorphone|dilaudid'), 1, 0)) opioid
  FROM `{E}.infusiondrug` WHERE infusionoffset BETWEEN 0 AND 2880 GROUP BY 1),
aps AS (SELECT patientunitstayid id, intubated, NULLIF(eyes,-1)+NULLIF(motor,-1)+NULLIF(verbal,-1) gcs FROM `{E}.apacheapsvar`),
apr AS (SELECT patientunitstayid id, MAX(apachescore) apache FROM `{E}.apachepatientresult`
  WHERE apacheversion = 'IVa' AND apachescore >= 0 GROUP BY 1),
etoh AS (SELECT DISTINCT patientunitstayid id FROM `{E}.pasthistory` WHERE REGEXP_CONTAINS(LOWER(pasthistorypath), r'etoh|alcohol'))
SELECT b.*,
  b.hospitalid IN (SELECT hospitalid FROM del_hosp) hosp_charts_delirium,
  b.id IN (SELECT id FROM anymed) has_meds,
  lev.lev_start, asm.asm_start,
  b.id IN (SELECT id FROM sz_hx) sz_hx, b.id IN (SELECT id FROM sz_dx) sz_dx,
  (SELECT MAX(pos) FROM dl WHERE dl.id = b.id AND dl.t BETWEEN 0 AND 2879) del_early,
  (SELECT MAX(pos) FROM dl WHERE dl.id = b.id AND dl.t BETWEEN 2880 AND 10080) del_out,
  (SELECT MAX(pos) FROM sym WHERE sym.id = b.id AND sym.t BETWEEN 0 AND 2879) sym_early,
  (SELECT MAX(pos) FROM sym WHERE sym.id = b.id AND sym.t BETWEEN 2880 AND 10080) sym_out,
  (SELECT MIN(v) FROM rass_raw r WHERE r.id = b.id AND r.t BETWEEN 0 AND 2879) rass_min_early,
  (SELECT MIN(v) FROM rass_raw r WHERE r.id = b.id AND r.t BETWEEN 2880 AND 10080) rass_min_out,
  inf.propofol, inf.benzo, inf.dex, inf.opioid, aps.intubated, aps.gcs, apr.apache,
  b.id IN (SELECT id FROM etoh) etoh
FROM base b LEFT JOIN lev ON lev.id = b.id LEFT JOIN asm ON asm.id = b.id LEFT JOIN inf ON inf.id = b.id
LEFT JOIN aps ON aps.id = b.id LEFT JOIN apr ON apr.id = b.id
"""
df = q(sql)

# ---------------------------------------------------------------- cohort flow
df["age_n"] = pd.to_numeric(df.age.replace({"> 89": "90"}), errors="coerce")
flow = [("ABI admission diagnosis (APACHE)", len(df))]
def step(label, keep):
    global df
    df = df[keep].copy(); flow.append((label, len(df)))
step("First ICU unit stay", df.unitvisitnumber == 1)
df = df.sort_values("id").drop_duplicates("uniquepid"); flow.append(("One stay per patient", len(df)))
step("Adult (>=18 y)", df.age_n >= 18)
step("Hospital charts delirium score", df.hosp_charts_delirium)
step("Stay has medication records", df.has_meds)
step("ICU stay >= 48 h", df.los_min >= 2880)
step("No seizure/epilepsy history", ~df.sz_hx)
step("No seizure diagnosis <= 48 h", ~df.sz_dx)
step("No other antiseizure medication <= 48 h", ~(df.asm_start.notna() & (df.asm_start <= 2880)))
df["lev"] = (df.lev_start.notna() & (df.lev_start <= 2880)).astype(int)
df["lev_late"] = (df.lev_start.notna() & (df.lev_start > 2880)).astype(int)
print("COHORT FLOW"); print(mask(pd.DataFrame(flow, columns=["step", "n"])).to_string(index=False))
print(f"\nExposure: levetiracetam <=48 h {mask(pd.DataFrame({'n':[int(df.lev.sum())]})).n[0]}, "
      f"none <=48 h {mask(pd.DataFrame({'n':[int((1-df.lev).sum())]})).n[0]} (late starters {mask(pd.DataFrame({'n':[int(df.lev_late.sum())]})).n[0]}); "
      f"hospitals {df.hospitalid.nunique()}")

# ---------------------------------------------------------------- variables
dx = df.dx.str.lower()
df["inj"] = np.select([dx.str.contains("subarachnoid"), dx.str.contains(r"hemorrhage/hematoma")], ["SAH", "ICH"], "TBI/SDH")
df["surgery"] = dx.str.contains("surgery").astype(int)
df["female"] = (df.gender == "Female").astype(int)
df["vent"] = df.intubated.fillna(0).clip(0, 1).astype(int)
for c in ["propofol", "benzo", "dex", "opioid"]: df[c] = df[c].fillna(0).astype(int)
df["etoh"] = df.etoh.astype(int)
df["del_early"] = df.del_early.fillna(0).astype(int)
df["coma_early"] = (df.rass_min_early <= -4).astype(int)
df["teach"] = (df.teachingstatus == True).astype(int)
df["yr2015"] = (df.yr == 2015).astype(int)
for c in ["gcs", "apache"]:
    df[c + "_miss"] = df[c].isna().astype(int); df[c] = df[c].fillna(df[c].median())
df["region"] = df.region.fillna("Unknown")
df["death"] = (df.hstat == "Expired").astype(int)
df["coma"] = (df.rass_min_out <= -4).astype(int)
X = pd.get_dummies(df[["age_n", "female", "inj", "surgery", "gcs", "gcs_miss", "apache", "apache_miss", "vent", "propofol",
                       "benzo", "dex", "opioid", "etoh", "del_early", "coma_early", "teach", "yr2015", "region"]],
                   columns=["inj", "region"], drop_first=True).astype(float)

def smd(x, t, w=None):
    w = np.ones(len(x)) if w is None else w
    m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
    v1, v0 = np.average((x[t == 1] - m1) ** 2, weights=w[t == 1]), np.average((x[t == 0] - m0) ** 2, weights=w[t == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2) if v1 + v0 > 0 else 0.0

def iptw(Xs, t):
    ps = LogisticRegression(max_iter=5000, C=1e6).fit(Xs, t).predict_proba(Xs)[:, 1]
    p = t.mean(); w = np.where(t == 1, p / ps, (1 - p) / (1 - ps))
    lo, hi = np.percentile(w, [1, 99]); return np.clip(w, lo, hi), ps

def fit(y, t, Xs, w=None, cov=False, cluster=None):
    Z = pd.concat([pd.Series(t, name="lev", index=Xs.index), Xs], axis=1) if cov else pd.DataFrame({"lev": t}, index=Xs.index)
    Z = sm.add_constant(Z, has_constant="add")
    m = sm.GLM(y, Z, family=sm.families.Binomial(), freq_weights=None, var_weights=w)
    r = m.fit(cov_type="cluster", cov_kwds={"groups": cluster}) if cluster is not None else m.fit(cov_type="HC1")
    b, se = r.params["lev"], r.bse["lev"]
    return np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se), r.pvalues["lev"]

def evalue(or_, lo):
    rr = or_; e = lambda r: r + np.sqrt(r * (r - 1)) if r >= 1 else 1
    return e(rr), (e(lo) if lo > 1 else 1.0)

def analyse(sub, ycol, label, rows, cov_cols=None):
    sub = sub[sub[ycol].notna()]
    t = sub.lev.values; y = sub[ycol].astype(int).values; Xs = X.loc[sub.index]
    Xs = Xs.loc[:, Xs.std() > 0]
    if cov_cols is not None: Xs = Xs[[c for c in Xs.columns if c in cov_cols]]
    w, ps = iptw(Xs, t); cl = sub.hospitalid.values
    u = fit(y, t, Xs, cluster=cl); wt = fit(y, t, Xs, w=w, cluster=cl); dr = fit(y, t, Xs, w=w, cov=True, cluster=cl)
    e1, e2 = evalue(*dr[:2])
    rows.append(dict(analysis=label, n=len(sub), ev_lev=int(y[t == 1].sum()), n_lev=int((t == 1).sum()),
                     ev_ctl=int(y[t == 0].sum()), n_ctl=int((t == 0).sum()),
                     unadj=f"{u[0]:.2f} ({u[1]:.2f}-{u[2]:.2f})", iptw=f"{wt[0]:.2f} ({wt[1]:.2f}-{wt[2]:.2f})",
                     iptw_cov=f"{dr[0]:.2f} ({dr[1]:.2f}-{dr[2]:.2f})", p=round(dr[3], 4), evalue=f"{e1:.2f}/{e2:.2f}",
                     maxSMD_after=round(max(abs(smd(Xs[c].values, t, w)) for c in Xs.columns), 3),
                     ps_range=f"{ps.min():.3f}-{ps.max():.3f}"))
    return w

# ---------------------------------------------------------------- baseline (aggregate)
print("\nBASELINE (levetiracetam <=48 h vs none)")
t_all = df.lev.values
base = []
for c, lab in [("age_n", "Age, mean"), ("female", "Female"), ("apache", "APACHE IV score, mean"), ("gcs", "GCS day 1, mean"),
               ("vent", "Ventilated day 1"), ("opioid", "Opioid infusion <=48 h"), ("propofol", "Propofol <=48 h"),
               ("del_early", "Delirium <=48 h"), ("surgery", "Neurosurgical admission")]:
    v = df[c].astype(float).values
    base.append(dict(var=lab, lev=round(v[t_all == 1].mean() * (1 if "mean" in lab else 100), 1),
                     no_lev=round(v[t_all == 0].mean() * (1 if "mean" in lab else 100), 1), smd=round(smd(v, t_all), 3)))
for g in ["TBI/SDH", "ICH", "SAH"]:
    v = (df.inj == g).astype(float).values
    base.append(dict(var=g + " (%)", lev=round(v[t_all == 1].mean() * 100, 1), no_lev=round(v[t_all == 0].mean() * 100, 1), smd=round(smd(v, t_all), 3)))
print(pd.DataFrame(base).to_string(index=False))

# ---------------------------------------------------------------- outcomes
rows = []
df["delirium"] = df.del_out                       # assessed 48-168 h only (NaN = not assessed)
df["del_or_coma"] = np.where(df.del_out.notna() | df.rass_min_out.notna(), ((df.del_out == 1) | (df.coma == 1)).astype(float), np.nan)
df["incident"] = np.where(df.del_early == 0, df.del_out, np.nan)
df["del_any_item"] = df[["del_out", "sym_out"]].max(axis=1)
analyse(df, "delirium", "Primary: delirium 48-168 h (assessed)", rows)
analyse(df, "del_or_coma", "Delirium or coma 48-168 h", rows)
analyse(df.assign(coma_f=np.where(df.rass_min_out.notna(), df.coma, np.nan)), "coma_f", "Coma (RASS <=-4) 48-168 h", rows)
analyse(df, "death", "In-hospital death", rows)
analyse(df, "incident", "Incident delirium (no delirium <=48 h)", rows)
analyse(df[df.lev_late == 0], "delirium", "Excluding late levetiracetam starters", rows)
analyse(df, "del_any_item", "Delirium incl. 'Symptoms of delirium' item", rows)
for g in ["TBI/SDH", "ICH", "SAH"]:
    try: analyse(df[df.inj == g], "delirium", f"Subgroup {g}", rows)
    except Exception as ex: rows.append(dict(analysis=f"Subgroup {g}", n=int((df.inj == g).sum()), unadj=f"not estimable ({type(ex).__name__})"))
res = pd.DataFrame(rows)
print("\nRESULTS (OR, 95% CI; cluster-robust SE by hospital; IPTW stabilized, 1/99 pct truncation)")
print(mask(res).to_string(index=False))
res_masked = mask(res); res_masked.to_csv("/content/eicu_replication_results.csv", index=False)
print("\nSaved aggregate results to /content/eicu_replication_results.csv")


In [ ]:
# Stage 7c — eICU robustness: overlap weights, hospital fixed effects, assessment intensity (aggregates only)
import numpy as np, pandas as pd, statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from statsmodels.discrete.conditional_models import ConditionalLogit
from google.cloud import bigquery as bq

# number of delirium assessments 48-168 h (screening intensity) per stay
ids = [int(i) for i in df.id]
cnt = client.query(f"""SELECT patientunitstayid id, COUNT(*) n_assess FROM `{E}.nursecharting`
  WHERE nursingchartcelltypevalname = 'Delirium Score' AND nursingchartoffset BETWEEN 2880 AND 10080
    AND patientunitstayid IN UNNEST(@ids) GROUP BY 1""",
  job_config=bq.QueryJobConfig(query_parameters=[bq.ArrayQueryParameter("ids", "INT64", ids)])).to_dataframe()
df = df.drop(columns=[c for c in ["n_assess"] if c in df]).merge(cnt, on="id", how="left")
df["n_assess"] = df.n_assess.fillna(0); df["log_assess"] = np.log1p(df.n_assess)
df.index = range(len(df))

CORE = ["age_n", "female", "surgery", "gcs", "gcs_miss", "apache", "apache_miss", "vent", "propofol", "benzo", "dex",
        "opioid", "etoh", "del_early", "coma_early"]
Xc = pd.get_dummies(df[CORE + ["inj"]], columns=["inj"], drop_first=True).astype(float)

# where is levetiracetam used? (aggregate)
hs = df.groupby("hospitalid").agg(n=("lev", "size"), lev=("lev", "sum"))
print(f"Hospitals: {len(hs)}; hospitals with any early levetiracetam: {(hs.lev > 0).sum()}; "
      f"share of treated patients in the 3 highest-use hospitals: {hs.lev.nlargest(3).sum() / hs.lev.sum():.0%}")
a = df[df.del_out.notna()]
print(f"Assessed 48-168 h: median delirium assessments, levetiracetam {a[a.lev==1].n_assess.median():.0f} vs comparator {a[a.lev==0].n_assess.median():.0f}")

def smd(x, t, w):
    m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
    v1, v0 = x[t == 1].var(), x[t == 0].var()
    return (m1 - m0) / np.sqrt((v1 + v0) / 2) if v1 + v0 > 0 else 0.0

def ow_fit(sub, ycol, extra=(), hosp_fe=False):
    sub = sub[sub[ycol].notna()]
    Xs = Xc.loc[sub.index].copy()
    for e in extra: Xs[e] = sub[e]
    if hosp_fe:
        hd = pd.get_dummies(sub.hospitalid.astype(str), prefix="h", drop_first=True).astype(float); Xs = pd.concat([Xs, hd], axis=1)
    Xs = Xs.loc[:, Xs.std() > 0]
    t = sub.lev.values; y = sub[ycol].astype(int).values
    ps = LogisticRegression(max_iter=5000, C=1.0).fit(StandardScaler().fit_transform(Xs), t).predict_proba(StandardScaler().fit_transform(Xs))[:, 1]
    w = np.where(t == 1, 1 - ps, ps)                      # overlap weights (ATO)
    Z = sm.add_constant(pd.concat([pd.Series(t, name="lev", index=Xs.index), Xs], axis=1), has_constant="add")
    r = sm.GLM(y, Z, family=sm.families.Binomial(), var_weights=w).fit(cov_type="cluster", cov_kwds={"groups": sub.hospitalid.values})
    b, se = r.params["lev"], r.bse["lev"]
    core_cols = [c for c in Xs.columns if not c.startswith("h_")]
    return dict(n=len(sub), n_lev=int(t.sum()), OR=f"{np.exp(b):.2f} ({np.exp(b-1.96*se):.2f}-{np.exp(b+1.96*se):.2f})",
                p=round(r.pvalues['lev'], 4), maxSMD=round(max(abs(smd(Xs[c].values, t, w)) for c in core_cols), 3))

def clogit(sub, ycol, extra=()):
  try:
    return _clogit(sub, ycol, extra)
  except Exception as ex:
    sub = sub[sub[ycol].notna()]
    Xs = Xc.loc[sub.index].copy()
    for e in extra: Xs[e] = sub[e]
    hd = pd.get_dummies(sub.hospitalid.astype(str), prefix="h", drop_first=True).astype(float)
    Z = sm.add_constant(pd.concat([pd.Series(sub.lev.values, name="lev", index=Xs.index), Xs, hd], axis=1).loc[:, lambda d: d.std() > 0], has_constant="add")
    r = sm.GLM(sub[ycol].astype(int).values, Z, family=sm.families.Binomial()).fit(cov_type="cluster", cov_kwds={"groups": sub.hospitalid.values})
    b, se = r.params["lev"], r.bse["lev"]
    return dict(n=len(sub), n_lev=int(sub.lev.sum()), OR=f"{np.exp(b):.2f} ({np.exp(b-1.96*se):.2f}-{np.exp(b+1.96*se):.2f})",
                p=round(r.pvalues['lev'], 4), maxSMD="(hospital-FE GLM; clogit failed)")

def _clogit(sub, ycol, extra=()):
    sub = sub[sub[ycol].notna()]
    Xs = Xc.loc[sub.index].copy()
    for e in extra: Xs[e] = sub[e]
    Xs = Xs.loc[:, Xs.std() > 0]
    Z = pd.concat([pd.Series(sub.lev.values, name="lev", index=Xs.index), Xs], axis=1)
    r = ConditionalLogit(sub[ycol].astype(int).values, Z, groups=sub.hospitalid.values).fit(disp=0)
    b, se = r.params["lev"], r.bse["lev"]
    return dict(n=len(sub), n_lev=int(sub.lev.sum()), OR=f"{np.exp(b):.2f} ({np.exp(b-1.96*se):.2f}-{np.exp(b+1.96*se):.2f})",
                p=round(r.pvalues['lev'], 4), maxSMD="")

rows = []
for y, lab in [("delirium", "Delirium 48-168 h (assessed)"), ("incident", "Incident delirium")]:
    rows.append(dict(outcome=lab, model="Overlap weights + covariates", **ow_fit(df, y)))
    rows.append(dict(outcome=lab, model="  + assessment intensity", **ow_fit(df, y, extra=("log_assess",))))
    rows.append(dict(outcome=lab, model="  + hospital fixed effects", **ow_fit(df, y, extra=("log_assess",), hosp_fe=True)))
    rows.append(dict(outcome=lab, model="Within-hospital conditional logit", **clogit(df, y, extra=("log_assess",))))
rows.append(dict(outcome="In-hospital death", model="Overlap weights + covariates", **ow_fit(df, "death")))
rows.append(dict(outcome="In-hospital death", model="Within-hospital conditional logit", **clogit(df, "death")))
out = pd.DataFrame(rows)
print("\nROBUSTNESS (OR, 95% CI; cluster-robust SE by hospital)"); print(mask(out).to_string(index=False))
mask(out).to_csv("/content/eicu_robustness.csv", index=False)


In [ ]:
# Stage 7d — restrict to hospitals that used early levetiracetam (aggregates only)
use = hs.index[hs.lev > 0]
sub = df[df.hospitalid.isin(use)]
print(f"Hospitals using levetiracetam: {len(use)}; stays {len(sub)}; treated {int(sub.lev.sum())}")
rows = []
for y, lab in [("delirium", "Delirium 48-168 h (assessed)"), ("incident", "Incident delirium"), ("death", "In-hospital death")]:
    rows.append(dict(outcome=lab, model="Overlap weights + covariates + assessment intensity",
                     **ow_fit(sub, y, extra=(("log_assess",) if y != "death" else ()))))
    rows.append(dict(outcome=lab, model="Hospital fixed effects (GLM)", **clogit(sub, y, extra=(("log_assess",) if y != "death" else ()))))
for g in ["TBI/SDH", "ICH"]:
    try: rows.append(dict(outcome=f"Delirium, {g}", model="Overlap weights + covariates", **ow_fit(df[df.inj == g], "delirium")))
    except Exception as ex: rows.append(dict(outcome=f"Delirium, {g}", model="not estimable"))
out2 = pd.DataFrame(rows); print(mask(out2).to_string(index=False))


In [ ]:
# Stage 8 — Harmonized analysis: same weighting and outcome definitions in MIMIC-IV and eICU, then pooled
# Aggregate output only (counts <10 masked).
import io, contextlib, numpy as np, pandas as pd, statsmodels.api as sm, warnings
from scipy.stats import norm, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from googleapiclient.discovery import build as gbuild
from google.cloud import bigquery as bq
warnings.filterwarnings("ignore")

# ---- keep the eICU frame, then add a CAM-ICU-only outcome so both cohorts use the same instrument
eicu = df.copy(); Xe = Xc.copy()
cam = client.query(f"""SELECT patientunitstayid id,
  MAX(IF(nursingchartoffset BETWEEN 0 AND 2879, IF(UPPER(TRIM(nursingchartvalue)) = 'YES', 1, 0), NULL)) cam_early,
  MAX(IF(nursingchartoffset BETWEEN 2880 AND 10080, IF(UPPER(TRIM(nursingchartvalue)) = 'YES', 1, 0), NULL)) cam_out
FROM `{E}.nursecharting`
WHERE nursingchartcelltypevalname = 'Delirium Score' AND UPPER(TRIM(nursingchartvalue)) IN ('YES', 'NO')
  AND patientunitstayid IN UNNEST(@ids) GROUP BY 1""",
  job_config=bq.QueryJobConfig(query_parameters=[bq.ArrayQueryParameter("ids", "INT64", [int(i) for i in eicu.id])])).to_dataframe()
eicu = eicu.merge(cam, on="id", how="left"); eicu.index = Xe.index
eicu["cam_early"] = eicu.cam_early.fillna(0)

# ---- rebuild the MIMIC-IV analytic frame with the stage-2 code (output suppressed)
PROJECT_ID = "YOUR_GCP_PROJECT"; WORK = "epi_lev_delirium"
HOSP = "physionet-data.mimiciv_3_1_hosp"; ICU = "physionet-data.mimiciv_3_1_icu"
def run(sql): client.query(sql).result()
src = open(f"{REPO}/scripts/stage2_lev_delirium.py").read()
src = src.split("# ---------------------------------------------------------------- 4. propensity score")[0]
with contextlib.redirect_stdout(io.StringIO()):
    exec(src)
mim = df.copy(); Xm = X.copy()
df = eicu  # leave df pointing at eICU as before
print(f"MIMIC-IV frame: {len(mim)} patients, {int(mim.lev_48h.sum())} treated; eICU frame: {len(eicu)} patients, {int(eicu.lev.sum())} treated")

# ---- one estimator for both cohorts: overlap weights from a penalized PS model + covariate-adjusted outcome model
def harm(d, Xall, t_col, y_col, keep, cluster=None):
    d = d[keep]; Xs = Xall.loc[d.index]; Xs = Xs.loc[:, Xs.std() > 0]
    t = d[t_col].astype(int).values; y = d[y_col].astype(int).values
    Z = StandardScaler().fit_transform(Xs)
    ps = LogisticRegression(C=1.0, max_iter=5000).fit(Z, t).predict_proba(Z)[:, 1]
    w = np.where(t == 1, 1 - ps, ps)
    D = sm.add_constant(pd.concat([pd.Series(t, index=Xs.index, name="lev"), Xs], axis=1), has_constant="add")
    m = sm.GLM(y, D, family=sm.families.Binomial(), var_weights=w)
    r = m.fit(cov_type="cluster", cov_kwds={"groups": cluster[keep.values] if hasattr(keep, "values") else cluster[keep]}) if cluster is not None else m.fit(cov_type="HC0")
    b, se = r.params["lev"], r.bse["lev"]
    def smd(x):
        m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
        s = np.sqrt((x[t == 1].var() + x[t == 0].var()) / 2); return 0 if s == 0 else (m1 - m0) / s
    return dict(n=len(d), n_lev=int(t.sum()), ev_lev=int(y[t == 1].sum()), ev_ctl=int(y[t == 0].sum()), n_ctl=int((t == 0).sum()),
                b=b, se=se, OR=f"{np.exp(b):.2f} ({np.exp(b-1.96*se):.2f}-{np.exp(b+1.96*se):.2f})",
                p=round(norm.sf(abs(b / se)) * 2, 4), maxSMD=round(max(abs(smd(Xs[c].values)) for c in Xs.columns), 3))

cl_e = eicu.hospitalid.values
specs = [
 # label, MIMIC (y, keep), eICU (y, keep)
 ("Delirium 48-168 h (assessed)",           ("delirium", mim.assessed == 1),                                   ("delirium", eicu.delirium.notna())),
 ("Incident delirium (none <=48 h)",        ("delirium", (mim.assessed == 1) & (mim.delirium_0_48 == 0)),     ("incident", eicu.incident.notna())),
 ("Delirium, CAM-ICU only",                 ("delirium", mim.assessed == 1),                                   ("cam_out", eicu.cam_out.notna())),
 ("Incident delirium, CAM-ICU only",        ("delirium", (mim.assessed == 1) & (mim.delirium_0_48 == 0)),     ("cam_out", eicu.cam_out.notna() & (eicu.cam_early == 0))),
 ("In-hospital death",                      ("hosp_death", mim.lev_48h.notna()),                               ("death", eicu.lev.notna())),
 ("Delirium, ICH only",                     ("delirium", (mim.assessed == 1) & (mim.injury == "ICH")),         ("delirium", eicu.delirium.notna() & (eicu.inj == "ICH"))),
 ("Delirium, TBI only (eICU: TBI/SDH)",     ("delirium", (mim.assessed == 1) & (mim.injury == "TBI")),         ("delirium", eicu.delirium.notna() & (eicu.inj == "TBI/SDH"))),
 ("Delirium, SAH only",                     ("delirium", (mim.assessed == 1) & (mim.injury == "SAH")),         ("delirium", eicu.delirium.notna() & (eicu.inj == "SAH"))),
]
rows = []
for lab, (ym, km), (ye, ke) in specs:
    res = {}
    for coh, f in [("MIMIC-IV", lambda: harm(mim, Xm, "lev_48h", ym, km)),
                   ("eICU", lambda: harm(eicu, Xe, "lev", ye, ke, cluster=cl_e))]:
        try: res[coh] = f()
        except Exception as ex: res[coh] = None; rows.append(dict(outcome=lab, cohort=coh, OR=f"not estimable ({type(ex).__name__})"))
        if res[coh]: rows.append(dict(outcome=lab, cohort=coh, **{k: v for k, v in res[coh].items() if k not in ("b", "se")}))
    if res.get("MIMIC-IV") and res.get("eICU"):
        b = np.array([res[c]["b"] for c in res]); se = np.array([res[c]["se"] for c in res]); wv = 1 / se ** 2
        m_ = (wv * b).sum() / wv.sum(); s_ = np.sqrt(1 / wv.sum()); Q = (wv * (b - m_) ** 2).sum()
        rows.append(dict(outcome=lab, cohort="Pooled (fixed effect)", n=sum(res[c]["n"] for c in res),
                         OR=f"{np.exp(m_):.2f} ({np.exp(m_-1.96*s_):.2f}-{np.exp(m_+1.96*s_):.2f})", p=round(2 * norm.sf(abs(m_ / s_)), 4),
                         maxSMD=f"I2={max(0, (Q-1)/Q)*100 if Q > 0 else 0:.0f}%, Q p={chi2.sf(Q, 1):.2f}"))
out = pd.DataFrame(rows).reindex(columns=["outcome", "cohort", "n", "n_lev", "ev_lev", "n_ctl", "ev_ctl", "OR", "p", "maxSMD"])
for c in ["n", "n_lev", "ev_lev", "n_ctl", "ev_ctl"]:
    out[c] = out[c].map(lambda v: int(v) if pd.notna(v) else "")
pd.set_option("display.width", 250)
print("\nHARMONIZED RESULTS: overlap weights (penalized PS) + covariate-adjusted logistic model; SE robust (MIMIC) or clustered by hospital (eICU)")
print(mask(out.fillna("")).to_string(index=False))
mask(out.fillna("")).to_csv("/content/harmonized_results.csv", index=False)


In [ ]:
# Stage 9 — Tests of confounding by indication (MIMIC-IV): active comparator, dose-response, negative controls, ICH severity
# Uses harm(), mim, Xm from stage 8. Aggregate output only (counts <10 masked).
import io, contextlib, numpy as np, pandas as pd, statsmodels.api as sm
from scipy.stats import norm
DER = HOSP.replace("_hosp", "_derived"); DELIRIUM_ITEM = 228332; RASS_ITEM = 228096
C1 = f"`{PROJECT_ID}.{WORK}.cohort_stage1`"; C2MAIN = f"`{PROJECT_ID}.{WORK}.cohort_stage2`"
C3 = f"`{PROJECT_ID}.{WORK}.cohort_active_comparator`"
ADMIN = "(LOWER(e.event_txt) LIKE '%administered%' OR e.event_txt IN ('Started', 'Restarted', 'Confirmed'))"
def show(title, d): print(f"\n{title}"); print(mask(d).to_string(index=False))
def fmt(r): return dict(n=r["n"], n_lev=r["n_lev"], ev_lev=r["ev_lev"], n_ctl=r["n_ctl"], ev_ctl=r["ev_ctl"], OR=r["OR"], p=r["p"], maxSMD=r["maxSMD"])

# =========================================================== A. ACTIVE COMPARATOR: levetiracetam vs phenytoin/fosphenytoin
run(f"""CREATE OR REPLACE TABLE {C3} AS
WITH seiz AS (SELECT DISTINCT hadm_id FROM `{HOSP}.diagnoses_icd`
  WHERE (icd_version = 9 AND (STARTS_WITH(icd_code, '345') OR STARTS_WITH(icd_code, '7803')))
     OR (icd_version = 10 AND REGEXP_CONTAINS(icd_code, r'^(G40|G41|R56)'))),
aud AS (SELECT DISTINCT hadm_id FROM `{HOSP}.diagnoses_icd`
  WHERE (icd_version = 9 AND REGEXP_CONTAINS(icd_code, r'^(291|303|3050)')) OR (icd_version = 10 AND STARTS_WITH(icd_code, 'F10'))),
pht AS (SELECT e.hadm_id, MIN(e.charttime) t FROM `{HOSP}.emar` e
  WHERE REGEXP_CONTAINS(LOWER(e.medication), r'phenytoin|fosphenytoin|dilantin|cerebyx') AND {ADMIN} GROUP BY 1),
oth AS (SELECT e.hadm_id, MIN(e.charttime) t FROM `{HOSP}.emar` e
  WHERE REGEXP_CONTAINS(LOWER(e.medication), r'lacosamide|vimpat|valpro|depakote|phenobarbital') AND {ADMIN} GROUP BY 1)
SELECT c.*, (sz.hadm_id IS NOT NULL) AS seizure_index_adm, (au.hadm_id IS NOT NULL) AS alcohol_use_disorder,
  (p.t IS NOT NULL AND p.t <= DATETIME_ADD(c.intime, INTERVAL 48 HOUR)) AS pht_48h,
  (o.t IS NOT NULL AND o.t <= DATETIME_ADD(c.intime, INTERVAL 48 HOUR)) AS oth_48h
FROM {C1} c LEFT JOIN seiz sz USING (hadm_id) LEFT JOIN aud au USING (hadm_id)
LEFT JOIN pht p USING (hadm_id) LEFT JOIN oth o USING (hadm_id)
WHERE c.in_cohort AND c.has_emar AND c.los >= 2 AND sz.hadm_id IS NULL""")
show("A1. Antiseizure prophylaxis within 48 h (no seizure code, ICU >= 48 h)", q(f"""SELECT
  COUNTIF(lev_48h AND NOT pht_48h AND NOT oth_48h) lev_only, COUNTIF(pht_48h AND NOT lev_48h AND NOT oth_48h) pht_only,
  COUNTIF(lev_48h AND pht_48h) both_lev_pht, COUNTIF(oth_48h) other_asm FROM {C3}"""))
show("A2. Levetiracetam-only vs phenytoin-only by admission era", q(f"""SELECT anchor_year_group era,
  COUNTIF(lev_48h AND NOT pht_48h AND NOT oth_48h) lev_only, COUNTIF(pht_48h AND NOT lev_48h AND NOT oth_48h) pht_only
  FROM {C3} GROUP BY 1 ORDER BY 1"""))
run(f"CREATE OR REPLACE TABLE {C3} AS SELECT * FROM {C3} WHERE NOT oth_48h AND (lev_48h OR pht_48h) AND NOT (lev_48h AND pht_48h)")

src2 = open(f"{REPO}/scripts/stage2_lev_delirium.py").read()
s_an = src2[src2.index("# ---------------------------------------------------------------- 2. analysis dataset"):
            src2.index("# ---------------------------------------------------------------- 4. propensity score")]
C2 = C3
with contextlib.redirect_stdout(io.StringIO()):
    exec(s_an)
ac = df.copy(); Xac = X.copy(); df = eicu; C2 = C2MAIN
rows = []
for lab, y, keep in [("Delirium 48-168 h (assessed)", "delirium", ac.assessed == 1),
                     ("Incident delirium", "delirium", (ac.assessed == 1) & (ac.delirium_0_48 == 0)),
                     ("In-hospital death", "hosp_death", ac.lev_48h.notna())]:
    try: rows.append(dict(outcome=lab, **fmt(harm(ac, Xac, "lev_48h", y, keep))))
    except Exception as ex: rows.append(dict(outcome=lab, OR=f"not estimable ({type(ex).__name__})"))
show("A3. ACTIVE COMPARATOR — levetiracetam (treated) vs phenytoin/fosphenytoin (comparator); overlap weights + covariates", pd.DataFrame(rows))

# =========================================================== B. DOSE-RESPONSE (main cohort)
dose = q(f"""WITH b AS (SELECT stay_id, hadm_id, intime, admittime FROM {C2MAIN}),
em AS (SELECT b.stay_id, e.emar_id FROM b JOIN `{HOSP}.emar` e ON e.hadm_id = b.hadm_id
  WHERE REGEXP_CONTAINS(LOWER(e.medication), r'levetiracetam|keppra') AND {ADMIN}
    AND e.charttime BETWEEN b.admittime AND DATETIME_ADD(b.intime, INTERVAL 48 HOUR)),
emd AS (SELECT em.stay_id, em.emar_id, MAX(SAFE_CAST(d.dose_given AS FLOAT64)) mg FROM em
  JOIN `{HOSP}.emar_detail` d ON d.emar_id = em.emar_id WHERE LOWER(d.dose_given_unit) = 'mg' GROUP BY 1, 2),
es AS (SELECT stay_id, SUM(mg) mg_emar FROM emd GROUP BY 1),
ie AS (SELECT b.stay_id, SUM(IF(LOWER(ie.amountuom) = 'mg', ie.amount, NULL)) mg_ie FROM b
  JOIN `{ICU}.inputevents` ie ON ie.stay_id = b.stay_id JOIN `{ICU}.d_items` di ON di.itemid = ie.itemid
  WHERE REGEXP_CONTAINS(LOWER(di.label), r'levetiracetam|keppra') AND ie.starttime <= DATETIME_ADD(b.intime, INTERVAL 48 HOUR) GROUP BY 1)
SELECT b.stay_id, GREATEST(IFNULL(es.mg_emar, 0), IFNULL(ie.mg_ie, 0)) mg48
FROM b LEFT JOIN es USING (stay_id) LEFT JOIN ie USING (stay_id)""")
md = mim.merge(dose, on="stay_id", how="left"); md.index = mim.index; md["mg48"] = md.mg48.fillna(0)
tr = md[md.lev_48h == 1]
med = float(tr.mg48[tr.mg48 > 0].median())
print(f"\nB1. Cumulative levetiracetam dose in first 48 h among treated: median {med:.0f} mg "
      f"(IQR {tr.mg48[tr.mg48 > 0].quantile(.25):.0f}-{tr.mg48[tr.mg48 > 0].quantile(.75):.0f}); "
      f"treated with no recorded mg dose: {mask(pd.DataFrame({'n':[int((tr.mg48 == 0).sum())]})).n[0]}")
md["dose_cat"] = np.select([md.lev_48h == 0, md.mg48 <= med], ["none", "low"], "high")
md.loc[(md.lev_48h == 1) & (md.mg48 == 0), "dose_cat"] = "unknown"
a = md[(md.assessed == 1) & (md.dose_cat != "unknown")]
Xa = Xm.loc[a.index]; Xa = Xa.loc[:, Xa.std() > 0]
D = sm.add_constant(pd.concat([pd.get_dummies(a.dose_cat, dtype=float)[["low", "high"]], Xa], axis=1), has_constant="add")
r = sm.GLM(a.delirium.astype(int).values, D, family=sm.families.Binomial()).fit(cov_type="HC0")
cats = []
for c in ["none", "low", "high"]:
    s = a[a.dose_cat == c]
    o = "1 (reference)" if c == "none" else f"{np.exp(r.params[c]):.2f} ({np.exp(r.params[c]-1.96*r.bse[c]):.2f}-{np.exp(r.params[c]+1.96*r.bse[c]):.2f})"
    cats.append(dict(dose=c + ("" if c == "none" else (f" (<= {med:.0f} mg)" if c == "low" else f" (> {med:.0f} mg)")), n=len(s), events=int(s.delirium.sum()),
                     pct=f"{100*s.delirium.mean():.1f}%", adj_OR=o))
show("B2. Delirium 48-168 h by cumulative 48-h dose (covariate-adjusted logistic, robust SE)", pd.DataFrame(cats))
t_ = a[a.dose_cat != "none"].copy(); t_["per_g"] = t_.mg48 / 1000
Xt = Xm.loc[t_.index]; Xt = Xt.loc[:, Xt.std() > 0]
for lab, sub in [("all treated", t_), ("treated, no delirium <= 48 h", t_[t_.delirium_0_48 == 0])]:
    Xs = Xt.loc[sub.index]; Xs = Xs.loc[:, Xs.std() > 0]
    rr = sm.GLM(sub.delirium.astype(int).values, sm.add_constant(pd.concat([sub.per_g, Xs], axis=1), has_constant="add"),
                family=sm.families.Binomial()).fit(cov_type="HC0")
    b, se = rr.params["per_g"], rr.bse["per_g"]
    print(f"B3. Dose trend among {lab} (n={len(sub)}): adjusted OR per 1000 mg {np.exp(b):.2f} ({np.exp(b-1.96*se):.2f}-{np.exp(b+1.96*se):.2f}), p={2*norm.sf(abs(b/se)):.3f}")

# =========================================================== C. NEGATIVE CONTROLS (main cohort)
nc = q(f"""WITH b AS (SELECT stay_id, hadm_id, intime, admittime FROM {C2MAIN}),
ppi AS (SELECT DISTINCT b.stay_id FROM b JOIN `{HOSP}.emar` e ON e.hadm_id = b.hadm_id
  WHERE REGEXP_CONTAINS(LOWER(e.medication), r'pantoprazole|omeprazole|esomeprazole|lansoprazole') AND {ADMIN}
    AND e.charttime BETWEEN b.admittime AND DATETIME_ADD(b.intime, INTERVAL 48 HOUR)),
dx AS (SELECT hadm_id,
    MAX(IF((icd_version = 9 AND icd_code = '5990') OR (icd_version = 10 AND icd_code = 'N390'), 1, 0)) uti,
    MAX(IF((icd_version = 9 AND STARTS_WITH(icd_code, '7070')) OR (icd_version = 10 AND STARTS_WITH(icd_code, 'L89')), 1, 0)) pressure_injury
  FROM `{HOSP}.diagnoses_icd` GROUP BY 1)
SELECT b.stay_id, IF(p.stay_id IS NULL, 0, 1) ppi_48h, IFNULL(dx.uti, 0) uti, IFNULL(dx.pressure_injury, 0) pressure_injury
FROM b LEFT JOIN ppi p USING (stay_id) LEFT JOIN dx USING (hadm_id)""")
mn = mim.merge(nc, on="stay_id", how="left"); mn.index = mim.index
Xn = Xm.copy(); Xn["lev_48h"] = mn.lev_48h.astype(float)
rows = []
rows.append(dict(control="Exposure: proton-pump inhibitor <= 48 h -> delirium (adjusted also for levetiracetam)",
                 **fmt(harm(mn, Xn, "ppi_48h", "delirium", mn.assessed == 1))))
for oc, lab in [("uti", "Outcome: urinary tract infection code"), ("pressure_injury", "Outcome: pressure injury code")]:
    rows.append(dict(control=f"{lab} <- levetiracetam <= 48 h", **fmt(harm(mn, Xm, "lev_48h", oc, mn.lev_48h.notna()))))
show("C. NEGATIVE CONTROLS (overlap weights + covariates). n_lev/ev_lev refer to the exposed group of each row", pd.DataFrame(rows))

# =========================================================== D. ICH SEVERITY (main cohort, ICH only)
sev = q(f"""WITH b AS (SELECT stay_id, hadm_id FROM {C2MAIN}),
pr AS (SELECT hadm_id, MAX(IF((icd_version = 9 AND REGEXP_CONTAINS(icd_code, r'^(012|013|021)'))
                          OR (icd_version = 10 AND REGEXP_CONTAINS(icd_code, r'^(009|00C|00B|00J)')), 1, 0)) neurosurg
  FROM `{HOSP}.procedures_icd` GROUP BY 1),
dx AS (SELECT hadm_id, MAX(icd_version) maxver,
  MAX(IF(icd_version = 10 AND icd_code = 'I615', 1, 0)) ivh FROM `{HOSP}.diagnoses_icd` GROUP BY 1)
SELECT b.stay_id, IFNULL(pr.neurosurg, 0) neurosurg, IFNULL(dx.ivh, 0) ivh, IF(dx.maxver = 10, 1, 0) icd10
FROM b LEFT JOIN pr USING (hadm_id) LEFT JOIN dx USING (hadm_id)""")
mi = mim.merge(sev, on="stay_id", how="left"); mi.index = mim.index
ich = mi[(mi.injury == "ICH")]
desc = []
for lab, c in [("Neurosurgical procedure", "neurosurg"), ("Intraventricular extension (ICD-10 era only)", "ivh")]:
    s = ich if c != "ivh" else ich[ich.icd10 == 1]
    desc.append(dict(marker=lab, levetiracetam=f"{100*s[s.lev_48h==1][c].mean():.1f}%", comparator=f"{100*s[s.lev_48h==0][c].mean():.1f}%"))
desc.append(dict(marker="First-day GCS, mean", levetiracetam=f"{ich[ich.lev_48h==1].gcs_min.mean():.1f}", comparator=f"{ich[ich.lev_48h==0].gcs_min.mean():.1f}"))
show("D1. ICH severity markers by exposure", pd.DataFrame(desc))
Xi = Xm.copy(); Xi["neurosurg"] = mi.neurosurg.astype(float); Xi["ivh_icd10"] = (mi.ivh * mi.icd10).astype(float); Xi["icd10_era"] = mi.icd10.astype(float)
rows = [dict(model="ICH, base covariates", **fmt(harm(mi, Xm, "lev_48h", "delirium", (mi.assessed == 1) & (mi.injury == "ICH")))),
        dict(model="ICH, + neurosurgery + intraventricular extension", **fmt(harm(mi, Xi, "lev_48h", "delirium", (mi.assessed == 1) & (mi.injury == "ICH")))),
        dict(model="ICH incident delirium, + severity markers", **fmt(harm(mi, Xi, "lev_48h", "delirium", (mi.assessed == 1) & (mi.injury == "ICH") & (mi.delirium_0_48 == 0))))]
show("D2. ICH: delirium 48-168 h (overlap weights + covariates)", pd.DataFrame(rows))

# =========================================================== E. eICU: is phenytoin recorded at all?
show("E. eICU phenytoin/fosphenytoin records (all stays)", q(f"""
SELECT 'medication' src, COUNT(*) n_rows, COUNT(DISTINCT patientunitstayid) stays FROM `{E}.medication`
  WHERE REGEXP_CONTAINS(LOWER(drugname), r'pheny|fosphen|dilantin|cerebyx')
UNION ALL SELECT 'infusiondrug', COUNT(*), COUNT(DISTINCT patientunitstayid) FROM `{E}.infusiondrug`
  WHERE REGEXP_CONTAINS(LOWER(drugname), r'pheny|fosphen|dilantin|cerebyx')
UNION ALL SELECT 'medication rows with blank drugname', COUNT(*), COUNT(DISTINCT patientunitstayid) FROM `{E}.medication`
  WHERE drugname IS NULL OR TRIM(drugname) = ''"""))


In [ ]:
# Stage 9b — neurosurgery as a confounder in the main MIMIC-IV analysis; corrected eICU phenytoin probe (aggregates only)
ns = q(f"""WITH b AS (SELECT stay_id, hadm_id, intime FROM {C2MAIN}),
pr AS (SELECT b.stay_id,
   MAX(IF(p.chartdate <= DATE(DATETIME_ADD(b.intime, INTERVAL 48 HOUR)), 1, 0)) neurosurg_48h,
   MAX(1) neurosurg_any
 FROM b JOIN `{HOSP}.procedures_icd` p ON p.hadm_id = b.hadm_id
 WHERE (p.icd_version = 9 AND REGEXP_CONTAINS(p.icd_code, r'^(012|013|021)'))
    OR (p.icd_version = 10 AND REGEXP_CONTAINS(p.icd_code, r'^(009|00C|00B|00J)'))
 GROUP BY 1)
SELECT b.stay_id, IFNULL(pr.neurosurg_48h, 0) neurosurg_48h, IFNULL(pr.neurosurg_any, 0) neurosurg_any FROM b LEFT JOIN pr USING (stay_id)""")
mz = mi.merge(ns, on="stay_id", how="left"); mz.index = mim.index
print("Neurosurgical procedure by 48 h: levetiracetam "
      f"{100*mz[mz.lev_48h==1].neurosurg_48h.mean():.1f}% vs comparator {100*mz[mz.lev_48h==0].neurosurg_48h.mean():.1f}%; "
      f"any time: {100*mz[mz.lev_48h==1].neurosurg_any.mean():.1f}% vs {100*mz[mz.lev_48h==0].neurosurg_any.mean():.1f}%")
Xz = Xm.copy(); Xz["neurosurg_48h"] = mz.neurosurg_48h.astype(float)
Xz2 = Xz.copy(); Xz2["ivh_icd10"] = (mz.ivh * mz.icd10).astype(float); Xz2["icd10_era"] = mz.icd10.astype(float)
rows = []
for lab, Xuse in [("base covariates (harmonized)", Xm), ("+ neurosurgery <= 48 h", Xz), ("+ neurosurgery <= 48 h + intraventricular extension", Xz2)]:
    rows.append(dict(outcome="Delirium 48-168 h", model=lab, **fmt(harm(mz, Xuse, "lev_48h", "delirium", mz.assessed == 1))))
    rows.append(dict(outcome="Incident delirium", model=lab, **fmt(harm(mz, Xuse, "lev_48h", "delirium", (mz.assessed == 1) & (mz.delirium_0_48 == 0)))))
rows.append(dict(outcome="Delirium 48-168 h", model="no neurosurgery at any time", **fmt(harm(mz, Xm, "lev_48h", "delirium", (mz.assessed == 1) & (mz.neurosurg_any == 0)))))
rows.append(dict(outcome="Incident delirium", model="no neurosurgery at any time", **fmt(harm(mz, Xm, "lev_48h", "delirium", (mz.assessed == 1) & (mz.delirium_0_48 == 0) & (mz.neurosurg_any == 0)))))
rows.append(dict(outcome="Delirium, ICH", model="+ neurosurgery <= 48 h + IVH", **fmt(harm(mz, Xz2, "lev_48h", "delirium", (mz.assessed == 1) & (mz.injury == "ICH")))))
show("MIMIC-IV main cohort: adjustment for neurosurgery (overlap weights + covariates)", pd.DataFrame(rows))
show("eICU phenytoin/fosphenytoin (exact names; all stays)", q(f"""
SELECT 'medication' src, COUNT(*) n_rows, COUNT(DISTINCT patientunitstayid) stays FROM `{E}.medication`
  WHERE REGEXP_CONTAINS(LOWER(drugname), r'phenytoin|fosphenytoin|dilantin|cerebyx') AND drugordercancelled = 'No'
UNION ALL SELECT 'infusiondrug', COUNT(*), COUNT(DISTINCT patientunitstayid) FROM `{E}.infusiondrug`
  WHERE REGEXP_CONTAINS(LOWER(drugname), r'phenytoin|fosphenytoin|dilantin|cerebyx')"""))


In [ ]:
# Stage 10 — Single primary model in both cohorts (aggregate output only)
# MIMIC-IV: overlap weights + covariates + neurosurgery <= 48 h.  eICU: overlap weights + covariates (already incl. neurosurgical admission).
import numpy as np, pandas as pd
from scipy.stats import norm, chi2
mz["delirium_or_coma"] = ((mz.delirium == 1) | (mz.coma == 1)).astype(int)
eicu["coma_f"] = np.where(eicu.rass_min_out.notna(), eicu.coma, np.nan)

def ev(or_, lo, hi):
    f = lambda r: r + np.sqrt(r * (r - 1))
    if or_ >= 1: return f(or_), (f(lo) if lo > 1 else 1.0)
    return f(1 / or_), (f(1 / hi) if hi < 1 else 1.0)

specs = [("Delirium 48-168 h (assessed) [primary]", ("delirium", mz.assessed == 1), ("delirium", eicu.delirium.notna())),
         ("Incident delirium", ("delirium", (mz.assessed == 1) & (mz.delirium_0_48 == 0)), ("incident", eicu.incident.notna())),
         ("Delirium or coma", ("delirium_or_coma", mz.lev_48h.notna()), ("del_or_coma", eicu.del_or_coma.notna())),
         ("Coma (RASS <= -4)", ("coma", mz.lev_48h.notna()), ("coma_f", eicu.coma_f.notna())),
         ("In-hospital death", ("hosp_death", mz.lev_48h.notna()), ("death", eicu.lev.notna()))]
rows = []
for lab, (ym, km), (ye, ke) in specs:
    rm = harm(mz, Xz, "lev_48h", ym, km); re_ = harm(eicu, Xe, "lev", ye, ke, cluster=cl_e)
    b = np.array([rm["b"], re_["b"]]); se = np.array([rm["se"], re_["se"]]); w = 1 / se ** 2
    m = (w * b).sum() / w.sum(); s = np.sqrt(1 / w.sum()); Q = (w * (b - m) ** 2).sum()
    tau2 = max(0, (Q - 1) / (w.sum() - (w ** 2).sum() / w.sum())); ws = 1 / (se ** 2 + tau2)
    mr = (ws * b).sum() / ws.sum(); sr = np.sqrt(1 / ws.sum())
    for coh, r in [("MIMIC-IV", rm), ("eICU", re_)]:
        o, l, h = np.exp(r["b"]), np.exp(r["b"] - 1.96 * r["se"]), np.exp(r["b"] + 1.96 * r["se"])
        e1, e2 = ev(o, l, h)
        rows.append(dict(outcome=lab, cohort=coh, n=r["n"], n_lev=r["n_lev"], ev_lev=r["ev_lev"], n_ctl=r["n_ctl"], ev_ctl=r["ev_ctl"],
                         pct_lev=f"{100*r['ev_lev']/r['n_lev']:.1f}%", pct_ctl=f"{100*r['ev_ctl']/r['n_ctl']:.1f}%",
                         OR=r["OR"], p=r["p"], evalue=f"{e1:.2f}/{e2:.2f}", maxSMD=r["maxSMD"]))
    o, l, h = np.exp(m), np.exp(m - 1.96 * s), np.exp(m + 1.96 * s); e1, e2 = ev(o, l, h)
    rows.append(dict(outcome=lab, cohort="Pooled FE", n=rm["n"] + re_["n"], OR=f"{o:.2f} ({l:.2f}-{h:.2f})", p=round(2 * norm.sf(abs(m / s)), 4),
                     evalue=f"{e1:.2f}/{e2:.2f}", maxSMD=f"I2={max(0,(Q-1)/Q)*100 if Q>0 else 0:.0f}% Qp={chi2.sf(Q,1):.2f}; RE {np.exp(mr):.2f} ({np.exp(mr-1.96*sr):.2f}-{np.exp(mr+1.96*sr):.2f})"))
out = pd.DataFrame(rows)
for c in ["n", "n_lev", "ev_lev", "n_ctl", "ev_ctl"]:
    out[c] = out[c].map(lambda v: int(v) if pd.notna(v) else "")
pd.set_option("display.width", 300); pd.set_option("display.max_colwidth", 80)
print("PRIMARY MODEL (overlap weights + covariates; MIMIC-IV also neurosurgery <= 48 h)")
print(mask(out.fillna("")).to_string(index=False))


In [ ]:
# Stage 11 — compact regional-plot data (public summary statistics only): max -log10(p) per 10-kb bin
import io, json, numpy as np, pandas as pd
from googleapiclient.discovery import build as gbuild
drv = gbuild("drive", "v3")
def dl(fid): return pd.read_csv(io.BytesIO(drv.files().get_media(fileId=fid).execute()), sep="\t", compression="gzip", low_memory=False)
eq = read_local("eqtl_catalogue_SV2A_CACNA1B.tsv.gz"); meta = read_local("delirium_meta_SV2A_CACNA1B_region.tsv.gz")
GENES = {"SV2A": ("1", 149903318, 149918572, "spinal"), "CACNA1B": ("9", 137877782, 138124624, "cerebell")}
W, BIN = 500_000, 10_000
out = {}
for g, (ch, s, e, tis) in GENES.items():
    x = eq[(eq.gene == g) & eq.tissue.astype(str).str.contains(tis, case=False) & eq.study.astype(str).str.contains("GTEx", case=False)].copy()
    print(g, len(x), sorted(eq[eq.gene == g].study.astype(str).unique())[:8], sorted(eq[eq.gene == g].tissue.astype(str).unique())[:25])
    x["pvalue"] = pd.to_numeric(x.pvalue, errors="coerce"); x["position"] = pd.to_numeric(x.position, errors="coerce")
    ds = x.groupby("dataset_id").pvalue.min().idxmin(); x = x[x.dataset_id == ds]
    m = meta[(meta.Chr.astype(str).str.replace("chr", "") == ch)].copy()
    m["p"] = pd.to_numeric(m.P_sterr, errors="coerce"); m["pos"] = pd.to_numeric(m.Pos, errors="coerce")
    lo, hi = s - W, e + W
    def binned(pos, p):
        d = pd.DataFrame({"b": (pos // BIN) * BIN + BIN // 2, "lp": -np.log10(p)}).dropna()
        d = d[(d.b >= lo) & (d.b <= hi)].groupby("b").lp.max().round(1)
        return [[int((k - lo) // 1000), float(v)] for k, v in d.items()]
    lead = x.loc[x.pvalue.idxmin()]
    out[g] = dict(chrom=ch, start=s, end=e, lo=lo, tissue=str(x.tissue.iloc[0]), eqtl=binned(x.position, x.pvalue),
                  delirium=binned(m.pos, m.p), lead_pos=int(lead.position), lead_lp=round(float(-np.log10(lead.pvalue)), 2),
                  n_eqtl=int(len(x)), n_del=int(((m.pos >= lo) & (m.pos <= hi)).sum()))
s = json.dumps(out, separators=(",", ":"))
print(len(s)); print(s)


In [ ]:
# Stage 12a — locate downloadable resources (no patient data)
import requests
cands = {
 "smr_bin": ["https://yanglab.westlake.edu.cn/software/smr/download/smr-1.4.2-linux-x86_64.zip",
             "https://github.com/jianyanglab/SMR/releases/download/v1.4.2/smr-1.4.2-linux-x86_64.zip",
             "https://github.com/JianYang-Lab/SMR/releases/download/v1.4.2/smr-1.4.2-linux-x86_64.zip"],
 "brainmeta": ["https://yanglab.westlake.edu.cn/data/SMR/BrainMeta_cis_eqtl_summary.tar.gz",
               "https://yanglab.westlake.edu.cn/software/smr/download/BrainMeta_cis_eqtl_summary.tar.gz",
               "https://yanglab.westlake.edu.cn/data/brainmeta/BrainMeta_cis_eqtl_summary.tar.gz",
               "https://yanglab.westlake.edu.cn/data/brainmeta/cis_eqtl/BrainMeta_cis_eqtl_summary.tar.gz",
               "https://yanglab.westlake.edu.cn/data/smr/BrainMeta_cis_eqtl_summary.tar.gz"],
 "kg_vcf_chr1": ["https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz.tbi"],
 "kg_panel": ["https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel"],
 "plink": ["https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20231211.zip"],
 "delirium_meta": ["https://datashare.ed.ac.uk/bitstreams/b3652f1a-38c0-4c7b-a135-e00522aa5bfe/download"]}
for k, urls in cands.items():
    for u in urls:
        try:
            r = requests.head(u, allow_redirects=True, timeout=30)
            print(f"{k:14s} {r.status_code} {r.headers.get('Content-Length','?'):>12} {u}")
        except Exception as ex:
            print(f"{k:14s} ERR {type(ex).__name__} {u}")
try:
    html = requests.get("https://yanglab.westlake.edu.cn/software/smr/", timeout=30).text
    import re
    print("\nLinks on SMR page:", sorted(set(re.findall(r'href="([^"]+(?:\.zip|\.tar\.gz|\.gz))"', html)))[:40])
except Exception as ex: print("SMR page:", ex)


In [ ]:
# Stage 12b — upgraded drug-target MR for SV2A / CACNA1B (public summary statistics only)
#  (1) instruments: eQTL Catalogue brain datasets + brain cell-type eQTLs (Bryois et al. 2022, 8 cell types)
#  (2) SMR test + HEIDI-type heterogeneity test (LD from 1000 Genomes EUR, GRCh38)
#  (3) multi-variant MR: clumped instruments (p < 5e-6, r2 < 0.1) -> IVW, weighted median, MR-Egger
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysam"], check=False)
import os, io, re, requests, numpy as np, pandas as pd, pysam
from scipy import stats
from googleapiclient.discovery import build as gbuild
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
D = "/content/gen"; os.makedirs(D, exist_ok=True)
drv = gbuild("drive", "v3")
def dl_drive(fid): return pd.read_csv(io.BytesIO(drv.files().get_media(fileId=fid).execute()), sep="\t", compression="gzip", low_memory=False)
GENES = {"SV2A": ("1", 149903318, 149918572), "CACNA1B": ("9", 137877782, 138124624)}
ENSG = {"SV2A": "ENSG00000159164", "CACNA1B": "ENSG00000148408"}
WIN = 500_000

# ---------------------------------------------------------------- outcome data (already extracted, GRCh38)
meta = read_local("delirium_meta_SV2A_CACNA1B_region.tsv.gz"); fgr = read_local("finngen_R12_F5_DELIRIUM_SV2A_CACNA1B_region.tsv.gz")
print("meta columns:", list(meta.columns)); print("finngen columns:", list(fgr.columns))
def outcome_table(df, chrom, pos, ea, oa, b, se, p, rsid=None):
    o = pd.DataFrame({"chrom": df[chrom].astype(str).str.replace("chr", ""), "pos": pd.to_numeric(df[pos], errors="coerce"),
                      "ea": df[ea].astype(str).str.upper(), "oa": df[oa].astype(str).str.upper(),
                      "by": pd.to_numeric(df[b], errors="coerce"), "sey": pd.to_numeric(df[se], errors="coerce"),
                      "py": pd.to_numeric(df[p], errors="coerce")})
    if rsid and rsid in df: o["rsid"] = df[rsid].astype(str)
    return o.dropna(subset=["pos", "by", "sey"])
OUT = {"Delirium meta-analysis": outcome_table(meta, "Chr", "Pos", "Allele1", "Allele2", "Effect", "StdErr", "P_sterr", "rsid"),
       "FinnGen R12": outcome_table(fgr, "#chrom", "pos", "alt", "ref", "beta", "sebeta", "pval", "rsids")}

# ---------------------------------------------------------------- exposure 1: eQTL Catalogue (GRCh38)
eq = read_local("eqtl_catalogue_SV2A_CACNA1B.tsv.gz")
for c in ["position", "beta", "se", "pvalue", "maf"]: eq[c] = pd.to_numeric(eq[c], errors="coerce")
EXP = []
for (g, st, ti, ds), s in eq.groupby(["gene", "study", "tissue", "dataset_id"]):
    if "molecular_trait_id" in s: s = s[s.molecular_trait_id == s.loc[s.pvalue.idxmin(), "molecular_trait_id"]]
    EXP.append(dict(gene=g, source=f"{st} {ti}", df=pd.DataFrame({"chrom": GENES[g][0], "pos": s.position.astype(int),
        "ea": s.alt.str.upper(), "oa": s.ref.str.upper(), "bx": s.beta, "sex": s.se, "px": s.pvalue, "rsid": s.get("rsid", "")})))

# ---------------------------------------------------------------- exposure 2: brain cell-type eQTLs (Bryois et al. 2022; Zenodo 6104982)
Z = "https://zenodo.org/records/6104982/files/"
sp = pd.read_csv(Z + "snp_pos.txt.gz?download=1", sep="\t", compression="gzip")
print("Bryois snp_pos columns:", list(sp.columns), "rows:", len(sp))
spc = {c.lower(): c for c in sp.columns}
CT = ["Excitatory.neurons", "Inhibitory.neurons", "Astrocytes", "Oligodendrocytes", "OPCs...COPs", "Microglia", "Endothelial.cells", "Pericytes"]
for g, (ch, s0, e0) in GENES.items():
    for ct in CT:
        rows = []
        for chunk in pd.read_csv(Z + f"{ct}.{ch}.gz?download=1", sep=r"\s+", header=None, compression="gzip", chunksize=2_000_000,
                                 names=["gene_id", "snp", "dist", "p", "beta"]):
            gid = chunk.gene_id.astype(str); rows.append(chunk[gid.str.startswith(g + "_") | gid.str.contains(ENSG[g]) | (gid == g)])
        x = pd.concat(rows); x.to_csv(f"{D}/bryois_{g}_{ct}.tsv", sep="\t", index=False)
        if not len(x): print(f"  {g} {ct}: gene not tested"); continue
        x = x.merge(sp, left_on="snp", right_on=sp.columns[0], how="inner")
        posc = "SNP_id_hg38"
        eac = next(c for c in sp.columns if "effect" in c.lower() or c.lower() in ("a1", "alt"))
        oac = next(c for c in sp.columns if "other" in c.lower() or c.lower() in ("a2", "ref"))
        z = stats.norm.isf(x.p.clip(lower=1e-300) / 2)
        pos_new = pd.to_numeric(x[posc].astype(str).str.extract(r":(\d+)")[0], errors="coerce")
        x = x.assign(_pos=pos_new)
        EXP.append(dict(gene=g, source=f"Bryois 2022 {ct.replace('...', '/').replace('.', ' ')}", build=posc,
                        df=pd.DataFrame({"chrom": ch, "pos": x._pos, "ea": x[eac].astype(str).str.upper(),
                                         "oa": x[oac].astype(str).str.upper(), "bx": x.beta, "sex": (x.beta.abs() / z).replace(0, np.nan), "px": x.p, "rsid": x.snp})))
print("Exposure datasets:", len(EXP))

# ---------------------------------------------------------------- LD reference: 1000 Genomes EUR, GRCh38
KG = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000_genomes_project/release/20190312_biallelic_SNV_and_INDEL/ALL.chr{c}.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
panel = pd.read_csv("https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel", sep="\t")
EUR = set(panel[panel.super_pop == "EUR"]["sample"])
LDG = {}
NEED = {g: set() for g in GENES}
for ex in EXP:
    d_ = ex["df"].dropna(subset=["pos", "px"]); NEED[ex["gene"]] |= set(d_[d_.px < 1.57e-3].pos.astype(int))
for g, (ch, s0, e0) in GENES.items():
    vf = pysam.VariantFile(KG.format(c=ch)); keep = [s for s in vf.header.samples if s in EUR]; vf.subset_samples(keep)
    ctg = ch if ch in vf.header.contigs else "chr" + ch
    G = {}
    for rec in vf.fetch(ctg, s0 - WIN - 1000, e0 + WIN + 1000):
        if rec.alts is None or len(rec.alts) != 1 or rec.pos not in NEED[g]: continue
        G[(rec.pos, rec.ref.upper(), rec.alts[0].upper())] = np.array([sum(a for a in smp["GT"] if a is not None) for smp in rec.samples.values()], float)
    LDG[g] = G; print(f"1000G EUR LD panel {g}: {len(G)} variants, {len(keep)} samples")

def align_ld(g, pos, ea, oa):
    """return genotype vector coded as dosage of `ea`, or None"""
    G = LDG[g]
    if (pos, oa, ea) in G: return G[(pos, oa, ea)]
    if (pos, ea, oa) in G: return 2 - G[(pos, ea, oa)]
    return None

def harmonise(x, o):
    m = x.merge(o, on=["chrom", "pos"])
    same = (m.ea_x == m.ea_y) & (m.oa_x == m.oa_y); flip = (m.ea_x == m.oa_y) & (m.oa_x == m.ea_y)
    m = m[same | flip].copy(); m.loc[flip[same | flip].values, "by"] *= -1
    pal = m.apply(lambda r: {r.ea_x, r.oa_x} in ({"A", "T"}, {"C", "G"}), axis=1)
    return m[~pal].rename(columns={"ea_x": "ea", "oa_x": "oa"}).drop_duplicates(["pos", "ea"]).reset_index(drop=True)

def rmat(g, m):
    vecs = [align_ld(g, int(r.pos), r.ea, r.oa) for r in m.itertuples()]
    ok = [i for i, v in enumerate(vecs) if v is not None and v.std() > 0]
    if not ok: return m.iloc[[]], np.zeros((0, 0))
    M = np.vstack([vecs[i] for i in ok]); return m.iloc[ok].reset_index(drop=True), np.corrcoef(M)

def smr_heidi(g, m):
    top = m.px.idxmin(); t = m.loc[top]
    zx, zy = t.bx / t.sex, t.by / t.sey
    T = zx ** 2 * zy ** 2 / (zx ** 2 + zy ** 2); b = t.by / t.bx; p_smr = stats.chi2.sf(T, 1); se = abs(b) / np.sqrt(T) if T > 0 else np.nan
    # HEIDI: SNPs with p_eQTL < 1.57e-3, 0.05 < r2(top) < 0.9, pruned at r2 > 0.9, up to 20
    c = m[m.px < 1.57e-3].copy()
    c, R = rmat(g, c)
    if not len(c) or t.pos not in set(c.pos): return dict(b_smr=b, se_smr=se, p_smr=p_smr, p_heidi=np.nan, n_heidi=0)
    it = int(np.where(c.pos.values == t.pos)[0][0]); r2t = R[it] ** 2
    cand = [i for i in np.argsort(c.px.values) if i != it and 0.05 < r2t[i] < 0.9]
    sel = []
    for i in cand:
        if all(R[i, j] ** 2 <= 0.9 for j in sel): sel.append(i)
        if len(sel) == 20: break
    if len(sel) < 3: return dict(b_smr=b, se_smr=se, p_smr=p_smr, p_heidi=np.nan, n_heidi=len(sel))
    idx = [it] + sel; bx, sx, by, sy = (c.loc[idx, k].values for k in ["bx", "sex", "by", "sey"]); Rr = R[np.ix_(idx, idx)]
    bxy = by / bx
    C = Rr * (np.outer(sy, sy) / np.outer(bx, bx) + np.outer(by * sx, by * sx) / np.outer(bx ** 2, bx ** 2))
    d = bxy[1:] - bxy[0]
    V = C[1:, 1:] - C[1:, [0]] - C[[0], 1:] + C[0, 0]
    stat = float(d @ np.linalg.pinv(V) @ d)
    return dict(b_smr=b, se_smr=se, p_smr=p_smr, p_heidi=stats.chi2.sf(stat, len(sel)), n_heidi=len(sel))

def multi_mr(g, m, pthr=5e-6, r2max=0.1):
    c = m[m.px < pthr].sort_values("px")
    c, R = rmat(g, c)
    if not len(c): return dict(n_iv=0)
    keep = []
    for i in range(len(c)):
        if all(R[i, j] ** 2 < r2max for j in keep): keep.append(i)
    v = c.iloc[keep].copy()
    v.loc[v.bx < 0, ["bx", "by"]] *= -1
    out = dict(n_iv=len(v))
    if len(v) < 2: return out
    w = v.bx ** 2 / v.sey ** 2; ratio = v.by / v.bx
    b_ivw = (v.bx * v.by / v.sey ** 2).sum() / w.sum(); se_ivw = 1 / np.sqrt(w.sum())
    Q = (((v.by - b_ivw * v.bx) / v.sey) ** 2).sum(); phi = max(1, Q / (len(v) - 1)); se_ivw *= np.sqrt(phi)
    out.update(ivw=b_ivw, ivw_se=se_ivw, ivw_p=2 * stats.norm.sf(abs(b_ivw / se_ivw)), Q_p=stats.chi2.sf(Q, len(v) - 1))
    if len(v) >= 3:
        def wmed(r, ww):
            o = np.argsort(r); r, ww = r[o], ww[o] / ww.sum(); cs = np.cumsum(ww) - ww / 2
            return np.interp(0.5, cs, r)
        se_r = v.sey / v.bx.abs(); wr = 1 / se_r ** 2
        bm = wmed(ratio.values, wr.values); rng = np.random.default_rng(1)
        boots = [wmed(rng.normal(v.by, v.sey) / rng.normal(v.bx, v.sex), wr.values) for _ in range(1000)]
        out.update(wmed=bm, wmed_se=np.std(boots), wmed_p=2 * stats.norm.sf(abs(bm / np.std(boots))))
        X = np.column_stack([np.ones(len(v)), v.bx]); W = np.diag(1 / v.sey ** 2)
        beta = np.linalg.solve(X.T @ W @ X, X.T @ W @ v.by); res = v.by - X @ beta
        s2 = max(1, (res ** 2 / v.sey ** 2).sum() / (len(v) - 2)); cov = np.linalg.inv(X.T @ W @ X) * s2
        out.update(egger=beta[1], egger_se=np.sqrt(cov[1, 1]), egger_p=2 * stats.t.sf(abs(beta[1] / np.sqrt(cov[1, 1])), len(v) - 2),
                   egger_int_p=2 * stats.t.sf(abs(beta[0] / np.sqrt(cov[0, 0])), len(v) - 2))
    return out

fmt = lambda b, se: f"{np.exp(b):.3f} ({np.exp(b-1.96*se):.3f}-{np.exp(b+1.96*se):.3f})" if pd.notna(b) and pd.notna(se) else ""
rows = []
for ex in EXP:
    g = ex["gene"]; ch, s0, e0 = GENES[g]
    x = ex["df"].dropna(subset=["pos", "bx", "sex", "px"]); x = x[(x.pos >= s0 - WIN) & (x.pos <= e0 + WIN)]
    if not len(x): continue
    lead_p = x.px.min(); lead_F = ((x.bx / x.sex) ** 2).max()
    for oname, o in OUT.items():
        m = harmonise(x, o)
        if len(m) < 3: continue
        r = dict(gene=g, exposure=ex["source"], outcome=oname, n_snps=len(m), lead_p=f"{lead_p:.1e}", lead_F=round(lead_F, 1))
        if lead_p < 5e-8 and m.px.min() < 5e-8:
            sh = smr_heidi(g, m); r.update(SMR_OR=fmt(sh["b_smr"], sh["se_smr"]), p_SMR=f"{sh['p_smr']:.3f}",
                                           p_HEIDI=("" if pd.isna(sh["p_heidi"]) else f"{sh['p_heidi']:.3f}"), n_HEIDI=sh["n_heidi"])
        mm = multi_mr(g, m)
        r.update(n_iv=mm.get("n_iv", 0), IVW_OR=fmt(mm.get("ivw"), mm.get("ivw_se")), p_IVW=(f"{mm['ivw_p']:.3f}" if "ivw_p" in mm else ""),
                 WM_OR=fmt(mm.get("wmed"), mm.get("wmed_se")), Egger_OR=fmt(mm.get("egger"), mm.get("egger_se")),
                 p_Egger_int=(f"{mm['egger_int_p']:.2f}" if "egger_int_p" in mm else ""), p_Q=(f"{mm['Q_p']:.2f}" if "Q_p" in mm else ""))
        rows.append(r)
res = pd.DataFrame(rows).fillna("")
res.to_csv(f"{D}/mr_upgrade_results.tsv", sep="\t", index=False)
print("\nUPGRADED MR (OR per 1-SD higher genetically proxied expression; SMR only where lead eQTL p < 5e-8)")
print(res.to_string(index=False))
print("\nInstrument availability (lead eQTL p < 5e-8) by gene:")
print(res.drop_duplicates(["gene", "exposure"]).assign(strong=lambda d: d.lead_p.astype(float) < 5e-8).groupby("gene").strong.agg(["sum", "count"]))


In [ ]:
# Stage 12c — positive control: APOE-region genes (strongest known delirium locus) through the same MR / SMR / HEIDI / coloc pipeline
import os, io, zlib, requests, numpy as np, pandas as pd, pysam
from scipy import stats
PC = {"APOE": ("19", 44905754, 44909393, "ENSG00000130203"), "APOC1": ("19", 44914247, 44919349, "ENSG00000130208"),
      "TOMM40": ("19", 44890569, 44903689, "ENSG00000130204")}
LO, HI = 44_400_000, 45_400_000

# ---- 1. outcome: FinnGen R12 delirium GWAS, chr19 window (public, tabix-indexed; GRCh38)
#      (the delirium meta-analysis file on Edinburgh DataShare refuses automated downloads, so FinnGen is the outcome here)
row = pd.Series(["F5_DELIRIUM"])
url_o = "https://storage.googleapis.com/finngen-public-data-r12/summary_stats/release/finngen_R12_F5_DELIRIUM.gz"
tb = pysam.TabixFile(url_o, index=url_o + ".tbi"); ctg = "19" if "19" in tb.contigs else "chr19"
hdr = [h.lstrip("#") for h in tb.header[-1].split("\t")] if tb.header else ["chrom","pos","ref","alt","rsids","nearest_genes","pval","mlogp","beta","sebeta","af_alt","af_alt_cases","af_alt_controls"]
m19 = pd.DataFrame([r.split("\t") for r in tb.fetch(ctg, LO, HI)], columns=hdr)
m19["chrom"] = "19"
o19 = outcome_table(m19, "chrom", "pos", "alt", "ref", "beta", "sebeta", "pval", "rsids")
top = o19.loc[o19.py.idxmin()]
print(f"Outcome {row.iloc[0]} chr19:{LO}-{HI}: {len(o19)} variants; lead p = {top.py:.1e} at {int(top.pos)} ({top.get('rsid', '')})")
apoe4 = o19[o19.pos == 44908684]
if len(apoe4): print("rs429358 (APOE e4):", apoe4[["ea","oa","by","sey","py"]].to_string(index=False))

# ---- 2. exposures: eQTL Catalogue brain datasets (tabix) + Bryois astrocytes/microglia
md = pd.read_csv("https://raw.githubusercontent.com/eQTL-Catalogue/eQTL-Catalogue-resources/master/data_tables/dataset_metadata_r8.tsv", sep="\t")
md = md[(md.quant_method == "ge") & md.tissue_label.str.contains("brain|cortex|DLPFC", case=False, na=False)]
md = md[md.study_label.isin(["GTEx", "ROSMAP", "CommonMind", "BrainSeq", "Braineac2"])].drop_duplicates("dataset_id").head(8)
EXP2 = []
API = "https://www.ebi.ac.uk/eqtl/api/v2/datasets/{}/associations"
for _, d in md.iterrows():
    n0 = len(EXP2)
    for g, (ch, s0, e0, ens) in PC.items():
        recs, start = [], 0
        try:
            while True:
                r = requests.get(API.format(d.dataset_id), params={"gene_id": ens, "size": 1000, "start": start}, timeout=60)
                if r.status_code != 200: break
                js = r.json()
                if not js: break
                recs += js; start += len(js)
                if len(js) < 1000: break
        except Exception as ex:
            print(f"  FAIL {d.study_label} {d.tissue_label} {g}: {str(ex)[:60]}")
        if not recs: continue
        s = pd.DataFrame(recs)
        for c in ["position", "beta", "se", "pvalue"]: s[c] = pd.to_numeric(s[c], errors="coerce")
        if "molecular_trait_id" in s: s = s[s.molecular_trait_id == s.loc[s.pvalue.idxmin(), "molecular_trait_id"]]
        EXP2.append(dict(gene=g, source=f"{d.study_label} {d.tissue_label}", df=pd.DataFrame({"chrom": "19", "pos": s.position.astype(int),
            "ea": s.alt.str.upper(), "oa": s.ref.str.upper(), "bx": s.beta, "sex": s.se, "px": s.pvalue})))
    print(f"  {d.study_label} {d.tissue_label}: {len(EXP2) - n0} genes")
for ct in ["Astrocytes", "Microglia"]:
    rows = []
    for chunk in pd.read_csv(Z + f"{ct}.19.gz?download=1", sep=r"\s+", header=None, compression="gzip", chunksize=2_000_000,
                             names=["gene_id", "snp", "dist", "p", "beta"]):
        gid = chunk.gene_id.astype(str); rows.append(chunk[gid.str.contains("|".join(v[3] for v in PC.values())) | gid.str.split("_").str[0].isin(list(PC))])
    x = pd.concat(rows).merge(sp, left_on="snp", right_on="SNP", how="inner")
    x["pos"] = pd.to_numeric(x.SNP_id_hg38.astype(str).str.extract(r":(\d+)")[0], errors="coerce")
    z = stats.norm.isf(x.p.clip(lower=1e-300) / 2)
    for g in PC:
        s = x[x.gene_id.astype(str).str.split("_").str[0] == g]
        if not len(s): continue
        zz = stats.norm.isf(s.p.clip(lower=1e-300) / 2)
        EXP2.append(dict(gene=g, source=f"Bryois 2022 {ct}", df=pd.DataFrame({"chrom": "19", "pos": s.pos, "ea": s.effect_allele.astype(str).str.upper(),
            "oa": s.other_allele.astype(str).str.upper(), "bx": s.beta, "sex": (s.beta.abs() / zz).replace(0, np.nan), "px": s.p})))
print("Positive-control exposure datasets:", len(EXP2))

# ---- 3. LD panel chr19 (1000G EUR, GRCh38)
need = set()
for ex in EXP2:
    d_ = ex["df"].dropna(subset=["pos", "px"]); need |= set(d_[d_.px < 1.57e-3].pos.astype(int))
vf = pysam.VariantFile(KG.format(c="19")); keep = [s for s in vf.header.samples if s in EUR]; vf.subset_samples(keep)
ctg = "19" if "19" in vf.header.contigs else "chr19"; G = {}
for rec in vf.fetch(ctg, LO, HI):
    if rec.alts is None or len(rec.alts) != 1 or rec.pos not in need: continue
    G[(rec.pos, rec.ref.upper(), rec.alts[0].upper())] = np.array([sum(a for a in smp["GT"] if a is not None) for smp in rec.samples.values()], float)
for g in PC: LDG[g] = G
print(f"1000G EUR LD panel chr19: {len(G)} variants")

# ---- 4. coloc (same priors as the main analysis)
def log_abf(b, se, W):
    V = se ** 2; z = b / se; r = W ** 2 / (W ** 2 + V); return 0.5 * (np.log(1 - r) + r * z ** 2)
def lse(v): mx = np.max(v); return mx + np.log(np.sum(np.exp(v - mx)))
def coloc(l1, l2, p1=1e-4, p2=1e-4, p12=1e-5):
    a, b_ = lse(l1), lse(l2)
    lH3 = np.log(p1) + np.log(p2) + a + b_ + np.log1p(-np.exp(min(0, lse(l1 + l2) - a - b_)))
    v = np.array([0, np.log(p1) + a, np.log(p2) + b_, lH3, np.log(p12) + lse(l1 + l2)]); return np.exp(v - lse(v))

rows = []
for ex in EXP2:
    g = ex["gene"]; x = ex["df"].dropna(subset=["pos", "bx", "sex", "px"])
    m = harmonise(x, o19)
    if len(m) < 50: continue
    t = m.loc[m.px.idxmin()]
    r = dict(gene=g, exposure=ex["source"], n_snps=len(m), lead_eqtl_p=f"{t.px:.1e}")
    if t.px < 5e-8:
        sh = smr_heidi(g, m)
        r.update(SMR_OR=fmt(sh["b_smr"], sh["se_smr"]), p_SMR=f"{sh['p_smr']:.1e}", p_HEIDI=("" if pd.isna(sh["p_heidi"]) else f"{sh['p_heidi']:.1e}"))
    mm = multi_mr(g, m)
    r.update(n_iv=mm.get("n_iv", 0), IVW_OR=fmt(mm.get("ivw"), mm.get("ivw_se")), p_IVW=(f"{mm['ivw_p']:.1e}" if "ivw_p" in mm else ""))
    pp = coloc(log_abf(m.bx.values, m.sex.values, 0.15), log_abf(m.by.values, m.sey.values, 0.2))
    r.update(PP_H3=round(pp[3], 3), PP_H4=round(pp[4], 3))
    rows.append(r)
pc = pd.DataFrame(rows).fillna("")
pc.to_csv(f"{D}/positive_control_results.tsv", sep="\t", index=False)
print("\nPOSITIVE CONTROL (APOE region -> FinnGen R12 delirium)"); print(pc.to_string(index=False))


In [ ]:
# Stage 12d — positive control, bulk-brain eQTL instruments (eQTL Catalogue tabix), reusing o19 (FinnGen R12 F5_DELIRIUM) and EXP2 (Bryois)
import time
md = pd.read_csv("https://raw.githubusercontent.com/eQTL-Catalogue/eQTL-Catalogue-resources/master/data_tables/dataset_metadata_r8.tsv", sep="\t")
md = md[(md.quant_method == "ge") & md.tissue_label.str.contains("brain", case=False, na=False)]
pref = ["ROSMAP", "BrainSeq", "CommonMind"]
md = pd.concat([md[md.study_label.isin(pref)], md[md.study_label.str.startswith("GTEx") & md.tissue_label.str.contains("cortex|DLPFC|hippocampus|cerebellum", case=False)]]).drop_duplicates("dataset_id")
print(md[["study_label", "tissue_label", "dataset_id", "sample_size"]].to_string(index=False))
EXP3 = [e for e in EXP2 if e["source"].startswith("Bryois")]
for _, d in md.iterrows():
    url = f"https://ftp.ebi.ac.uk/pub/databases/spot/eQTL/sumstats/{d.study_id}/{d.dataset_id}/{d.dataset_id}.all.tsv.gz"
    t0 = time.time()
    try:
        tb = pysam.TabixFile(url); ctg = "19" if "19" in tb.contigs else "chr19"
        recs = [r.split("\t") for r in tb.fetch(ctg, LO, HI)]
        hdr = zlib.decompressobj(16 + zlib.MAX_WBITS).decompress(requests.get(url, headers={"Range": "bytes=0-65535"}, timeout=60).content).decode(errors="ignore").split("\n")[0].split("\t")
        x = pd.DataFrame(recs, columns=hdr[:len(recs[0])])
        k = 0
        for g, (ch, s0, e0, ens) in PC.items():
            s = x[x.gene_id == ens].copy()
            if not len(s): continue
            for c in ["position", "beta", "se", "pvalue"]: s[c] = pd.to_numeric(s[c], errors="coerce")
            if "molecular_trait_id" in s: s = s[s.molecular_trait_id == s.loc[s.pvalue.idxmin(), "molecular_trait_id"]]
            EXP3.append(dict(gene=g, source=f"{d.study_label} {d.tissue_label}", df=pd.DataFrame({"chrom": "19", "pos": s.position.astype(int),
                "ea": s.alt.str.upper(), "oa": s.ref.str.upper(), "bx": s.beta, "sex": s.se, "px": s.pvalue}))); k += 1
        print(f"  ok {d.study_label} {d.tissue_label}: {k} genes ({time.time() - t0:.0f}s)")
    except Exception as ex:
        print(f"  FAIL {d.study_label} {d.tissue_label}: {str(ex)[:80]}")

need = set()
for ex in EXP3:
    d_ = ex["df"].dropna(subset=["pos", "px"]); need |= set(d_[d_.px < 1.57e-3].pos.astype(int))
need.add(44908684)
vf = pysam.VariantFile(KG.format(c="19")); keep = [s for s in vf.header.samples if s in EUR]; vf.subset_samples(keep)
ctg = "19" if "19" in vf.header.contigs else "chr19"; G = {}
for rec in vf.fetch(ctg, LO, HI):
    if rec.alts is None or len(rec.alts) != 1 or rec.pos not in need: continue
    G[(rec.pos, rec.ref.upper(), rec.alts[0].upper())] = np.array([sum(a for a in smp["GT"] if a is not None) for smp in rec.samples.values()], float)
for g in PC: LDG[g] = G
print(f"1000G EUR LD panel chr19: {len(G)} variants")

rows = []
for ex in EXP3:
    g = ex["gene"]; x = ex["df"].dropna(subset=["pos", "bx", "sex", "px"])
    m = harmonise(x, o19)
    if len(m) < 50: continue
    t = m.loc[m.px.idxmin()]
    r = dict(gene=g, exposure=ex["source"], n_snps=len(m), lead_eqtl_p=f"{t.px:.1e}", lead_F=round((t.bx / t.sex) ** 2, 1))
    if t.px < 5e-8:
        sh = smr_heidi(g, m)
        r.update(SMR_OR=fmt(sh["b_smr"], sh["se_smr"]), p_SMR=f"{sh['p_smr']:.1e}", p_HEIDI=("" if pd.isna(sh["p_heidi"]) else f"{sh['p_heidi']:.2f}"), n_HEIDI=sh["n_heidi"])
    mm = multi_mr(g, m)
    r.update(n_iv=mm.get("n_iv", 0), IVW_OR=fmt(mm.get("ivw"), mm.get("ivw_se")), p_IVW=(f"{mm['ivw_p']:.1e}" if "ivw_p" in mm else ""),
             WM_OR=fmt(mm.get("wmed"), mm.get("wmed_se")), p_Egger_int=(f"{mm['egger_int_p']:.2f}" if "egger_int_p" in mm else ""))
    pp = coloc(log_abf(m.bx.values, m.sex.values, 0.15), log_abf(m.by.values, m.sey.values, 0.2))
    r.update(PP_H3=round(pp[3], 3), PP_H4=round(pp[4], 3))
    rows.append(r)
pc = pd.DataFrame(rows).fillna("")
pc.to_csv(f"{D}/positive_control_results.tsv", sep="\t", index=False)
print("\nPOSITIVE CONTROL (APOE-region expression -> FinnGen R12 delirium; OR per 1-SD higher expression)")
print(pc.to_string(index=False))


In [ ]:
# Software versions used in this analysis
import sys, platform, importlib
mods = ["pandas","numpy","scipy","statsmodels","sklearn","pysam","networkx","matplotlib","google.cloud.bigquery","requests","gseapy"]
print("Python", sys.version.split()[0], "|", platform.platform())
for m in mods:
    try: print(m, importlib.import_module(m).__version__)
    except Exception: print(m, "not available")
